# AutoResearch v2 - Momentum Alpha Discovery (T4x2)

This guarded stage evaluates deterministic/classical momentum baselines plus
program-evolution candidates on 100 US equities. If an LLM stage is enabled,
the report records loaded model id, execution state, and call count explicitly.

**Key differences from v1:**
- Model: Qwen2.5-Coder-32B-Instruct can run in 4-bit, with 14B fallback for T4x2 stability
- Universe: ~100 liquid US equities (vs 20)
- Reflection: reported only when recorded LLM/reflection calls actually execute
- Scoring: annual consistency (not just aggregate Sharpe)

**Kaggle setup:** Accelerator = **GPU T4 x2**, Internet = **On**


## Cell 1 â€” Install dependencies

In [ ]:
!pip install -q yfinance bitsandbytes "transformers>=4.44" "accelerate>=0.33"

## Cell 2 â€” Imports, configuration, paths

In [ ]:
import os, sys, json, re, time, random, gc, ast, traceback, hashlib
import threading as _th
import inspect as _inspect
from pathlib import Path
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
import numpy as np, pandas as pd, torch
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import yfinance as yf
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# â”€â”€ Reproducibility â”€â”€
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

def _env_truthy(value, default=False):
    if value is None:
        return default
    return str(value).strip().lower() in {"1", "true", "yes", "on"}

def _read_dotenv_value(path, *names):
    try:
        p = Path(path)
        if not p.exists():
            return None, "none"
        for raw_line in p.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            if key not in names:
                continue
            value = value.strip().strip('"').strip("'")
            if value:
                return value, f"dotenv:{key}"
    except Exception:
        pass
    return None, "none"

def _resolve_kaggle_secret(*names):
    try:
        from kaggle_secrets import UserSecretsClient
        client = UserSecretsClient()
        for name in names:
            try:
                value = client.get_secret(name)
            except Exception:
                value = None
            if value:
                return value, f"kaggle:{name}"
    except Exception:
        pass
    return None, "none"

def _resolve_token(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value, f"env:{name}"
    value, source = _resolve_kaggle_secret(*names)
    if value:
        return value, source
    value, source = _read_dotenv_value(".env", *names)
    if value:
        return value, source
    return None, "none"

def token_presence_text(name, value, source):
    state = "present" if value else "missing"
    return f"{name}: {state} (source={source})"

HF_TOKEN, HF_TOKEN_SOURCE = _resolve_token(
    "HF_TOKEN",
    "HUGGINGFACEHUB_API_TOKEN",
    "HUGGINGFACE_TOKEN",
    "HF_API_TOKEN",
)
HF_TOKEN = "hf_xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx"
HF_TOKEN_SOURCE = "hardcoded"

# â”€â”€ Paths â”€â”€
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)
RUN_ID = time.strftime("%Y%m%d-%H%M%S")
RUN_PROFILE = (os.getenv("AUTORESEARCH_RUN_PROFILE", "llm_research").strip()
               or "llm_research")
BENCHMARK_MODE = _env_truthy(os.getenv("AUTORESEARCH_BENCHMARK_MODE"), default=False)
ROADMAP_METHOD_FAMILIES = [
    "deterministic",
    "classical_risk_managed",
    "autoresearch_evolution",
    "lstm_sharpe",
    "tabular_ml",
    "gbt",
    "combination_studies",
]
EXECUTED_METHOD_FAMILIES = ["deterministic", "classical_risk_managed", "autoresearch_evolution"]
DEFERRED_METHOD_FAMILIES = [
    family for family in ROADMAP_METHOD_FAMILIES
    if family not in EXECUTED_METHOD_FAMILIES
]
ACTIVE_RESULT_SOURCES = ["deterministic", "llm_autoresearch", "parameter_search_evolution"]
ACTIVE_EXECUTION_SCOPE = "deterministic/classical/strict-llm autoresearch only"
FAMILY_EXECUTION_STATUS = {
    family: ("executed" if family in EXECUTED_METHOD_FAMILIES else "deferred")
    for family in ROADMAP_METHOD_FAMILIES
}
REPORT_SCOPE = f"{RUN_PROFILE}_{RUN_ID}"

def scoped_output(name, include_scope=True):
    stem = f"{REPORT_SCOPE}__{name}" if include_scope else name
    return OUT / stem

RESEARCH_LOG = scoped_output("research_log.jsonl")
PRICES_CACHE = OUT / "prices.parquet"
BEST_CODE    = scoped_output("best_signal.py")
SHARPE_PLOT  = scoped_output("sharpe_progress.png")
EQUITY_PLOT  = scoped_output("equity_curves.png")
SUMMARY_MD   = scoped_output("experiment_summary.md")
MEMO_FILE    = scoped_output("research_memo.txt")
RUNTIME_METADATA_FILE = scoped_output("runtime_metadata.json")
RUN_MANIFEST_FILE = scoped_output("run_manifest.json")
LATEST_RUNTIME_METADATA_FILE = OUT / "latest_runtime_metadata.json"
LATEST_RUN_MANIFEST_FILE = OUT / "latest_run_manifest.json"

# â”€â”€ Model â”€â”€
# Qwen2.5-Coder-32B at Q4 â‰ˆ 18 GB â†’ fits T4Ã—2 (32 GB combined).
# For faster runs, uncomment the 14B line below.
MODEL_VARIANT = (os.getenv("AUTORESEARCH_MODEL_VARIANT", "14b").strip().lower() or "14b")
CONFIGURED_MODEL_ID = (
    "Qwen/Qwen2.5-Coder-32B-Instruct"
    if MODEL_VARIANT in {"32b", "32", "large"}
    else "Qwen/Qwen2.5-Coder-14B-Instruct"
)
MODEL_ID = CONFIGURED_MODEL_ID
FALLBACK_MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"
CONFIGURED_FALLBACK_MODEL_ID = FALLBACK_MODEL_ID
ACTIVE_MODEL_ID = None

# â”€â”€ Loop config â”€â”€
N_BATCHES       = 25          # each batch â†’ 1 LLM call â†’ up to 3 candidates
REFLECT_EVERY   = 2           # reflection memo every N batches after first trigger
FIRST_REFLECTION_BATCH = int(os.getenv("AUTORESEARCH_FIRST_REFLECTION_BATCH", "2"))
SANDBOX_TIMEOUT = 30          # seconds per signal execution
TRAIN_END       = "2022-12-31"
LLM_PROMPT_MAX_TOKENS = int(os.getenv("AUTORESEARCH_PROMPT_MAX_TOKENS", "12000"))
LLM_BATCH_MAX_NEW_TOKENS = int(os.getenv("AUTORESEARCH_BATCH_MAX_NEW_TOKENS", "1100"))
LLM_REFLECTION_MAX_NEW_TOKENS = int(os.getenv("AUTORESEARCH_REFLECTION_MAX_NEW_TOKENS", "700"))
LLM_REPAIR_MAX_NEW_TOKENS = int(os.getenv("AUTORESEARCH_REPAIR_MAX_NEW_TOKENS", "900"))
EARLY_ABORT_NEGATIVE_SUCCESSFUL = int(os.getenv("AUTORESEARCH_EARLY_ABORT_NEGATIVE_SUCCESSFUL", "9"))
EARLY_ABORT_NEGATIVE_SHARPE = float(os.getenv("AUTORESEARCH_EARLY_ABORT_NEGATIVE_SHARPE", "-0.50"))
EARLY_ABORT_NEGATIVE_SCORE = float(os.getenv("AUTORESEARCH_EARLY_ABORT_NEGATIVE_SCORE", "-1.00"))
EARLY_ABORT_MIN_BATCH = int(os.getenv("AUTORESEARCH_EARLY_ABORT_MIN_BATCH", "2"))
EARLY_ABORT_MIN_REFLECTIONS = int(os.getenv("AUTORESEARCH_EARLY_ABORT_MIN_REFLECTIONS", "2"))
SEARCH_COLLAPSE_MIN_BATCHES = int(os.getenv("AUTORESEARCH_SEARCH_COLLAPSE_MIN_BATCHES", "3"))
SEARCH_COLLAPSE_FAILURE_RATE = float(os.getenv("AUTORESEARCH_SEARCH_COLLAPSE_FAILURE_RATE", "0.67"))
SEARCH_COLLAPSE_MIN_ERRORS = int(os.getenv("AUTORESEARCH_SEARCH_COLLAPSE_MIN_ERRORS", "6"))
STAGNATION_WINDOW_BATCHES = int(os.getenv("AUTORESEARCH_STAGNATION_WINDOW_BATCHES", "6"))
STAGNATION_MIN_SCORE_DELTA = float(os.getenv("AUTORESEARCH_STAGNATION_MIN_SCORE_DELTA", "0.10"))
STAGNATION_FAILURE_RATE = float(os.getenv("AUTORESEARCH_STAGNATION_FAILURE_RATE", "0.60"))
STAGNATION_MIN_FAILURES = int(os.getenv("AUTORESEARCH_STAGNATION_MIN_FAILURES", "8"))
STAGNATION_MIN_REFLECTIONS = int(os.getenv("AUTORESEARCH_STAGNATION_MIN_REFLECTIONS", "1"))
MIN_VIABLE_TRAIN_SHARPE = float(os.getenv("AUTORESEARCH_MIN_VIABLE_TRAIN_SHARPE", "0.00"))
MIN_VIABLE_RESEARCH_SCORE = float(os.getenv("AUTORESEARCH_MIN_VIABLE_RESEARCH_SCORE", "0.00"))
OBJECTIVE_MODE = (
    os.getenv(
        "AUTORESEARCH_OBJECTIVE_MODE",
        "market_neutral" if RUN_PROFILE == "llm_research" and not BENCHMARK_MODE else "benchmark_relative",
    ).strip().lower() or "market_neutral"
)
if OBJECTIVE_MODE not in {"market_neutral", "benchmark_relative"}:
    OBJECTIVE_MODE = "market_neutral"
PRIMARY_OBJECTIVE_LABEL = (
    "market-neutral portfolio Sharpe"
    if OBJECTIVE_MODE == "market_neutral"
    else "benchmark-relative active Sharpe"
)
DEFAULT_LLM_RESEARCH_MODE = (RUN_PROFILE == "llm_research" and not BENCHMARK_MODE)
RUN_BASELINE_SWEEP = True
RUN_DETERMINISTIC_SEARCH = True
RUN_LLM_STAGE   = _env_truthy(os.getenv("AUTORESEARCH_RUN_LLM_STAGE"), default=DEFAULT_LLM_RESEARCH_MODE)
RUN_MOE_STAGE   = _env_truthy(os.getenv("AUTORESEARCH_RUN_MOE_STAGE"), default=False)
RUN_LLM_SMOKE = _env_truthy(os.getenv("AUTORESEARCH_RUN_LLM_SMOKE"), default=False)
RUN_BNB_MODEL_LOAD = _env_truthy(
    os.getenv("AUTORESEARCH_ENABLE_BNB_LOAD"),
    default=(DEFAULT_LLM_RESEARCH_MODE or RUN_LLM_SMOKE),
)
RUN_PARAM_SEARCH = _env_truthy(os.getenv("AUTORESEARCH_RUN_PARAM_SEARCH"), default=False)
RUN_HELDOUT_EVAL = True
RUN_REPORTS = True
REFLECTION_ENABLED = bool(RUN_LLM_STAGE and REFLECT_EVERY > 0)

print(
    f"flags: baseline={RUN_BASELINE_SWEEP} deterministic={RUN_DETERMINISTIC_SEARCH} "
    f"param_search={RUN_PARAM_SEARCH} heldout={RUN_HELDOUT_EVAL} reports={RUN_REPORTS}"
)

# Structured parameter search constants must exist even when the stage is off:
# metric_cell_18 defines function defaults from these names at cell execution.
PARAM_SEARCH_TRIALS = 200
PARAM_SEARCH_SEED = SEED
PARAM_SEARCH_MODE = "random"
PARAM_SEARCH_SHORT_SPANS = [36, 39, 42, 45, 48, 51, 54, 57, 60, 63, 66]
PARAM_SEARCH_LONG_SPANS = [90, 95, 100, 105, 110, 120, 130, 140, 150]
PARAM_SEARCH_REGIME_THRESHOLDS = [18.0, 20.0, 22.0, 24.0, 28.0]
PARAM_SEARCH_VOL_WINDOWS = [10, 20, 30]
PARAM_SEARCH_VOL_GATE_WINDOWS = [10, 20, 40]

if DEFAULT_LLM_RESEARCH_MODE and not HF_TOKEN:
    print(
        "HF_TOKEN not found; attempting anonymous Hugging Face download. "
        "Add HF_TOKEN/HUGGINGFACEHUB_API_TOKEN only if the selected model requires auth."
    )

RUNTIME_STATE = {
    "run_id": RUN_ID,
    "run_profile": RUN_PROFILE,
    "report_scope": REPORT_SCOPE,
    "benchmark_mode": BENCHMARK_MODE,
    "configured_method_families": list(ROADMAP_METHOD_FAMILIES),
    "executed_method_families": list(EXECUTED_METHOD_FAMILIES),
    "deferred_method_families": list(DEFERRED_METHOD_FAMILIES),
    "family_execution_status": dict(FAMILY_EXECUTION_STATUS),
    "active_result_sources": list(ACTIVE_RESULT_SOURCES),
    "active_execution_scope": ACTIVE_EXECUTION_SCOPE,
    "configured_model_id": CONFIGURED_MODEL_ID,
    "configured_fallback_model_id": CONFIGURED_FALLBACK_MODEL_ID,
    "objective_mode": OBJECTIVE_MODE,
    "primary_objective_label": PRIMARY_OBJECTIVE_LABEL,
    "actual_model_id": None,
    "llm_stage_enabled": bool(RUN_LLM_STAGE),
    "moe_stage_enabled": bool(RUN_MOE_STAGE),
    "llm_smoke_enabled": bool(RUN_LLM_SMOKE),
    "llm_stage_loaded": False,
    "bnb_model_load_enabled": bool(RUN_BNB_MODEL_LOAD),
    "llm_stage_executed": False,
    "llm_calls": 0,
    "reflection_enabled": REFLECTION_ENABLED,
    "reflection_calls": 0,
    "structured_candidate_count": 0,
    "repair_batches": 0,
    "duplicate_rejections": 0,
    "semantic_family_rejections": 0,
    "format_failures": 0,
    "token_status": {
        "hf": {"present": bool(HF_TOKEN), "source": HF_TOKEN_SOURCE},
    },
}

ARTIFACT_PATHS = {
    "research_log": str(RESEARCH_LOG),
    "prices_cache": str(PRICES_CACHE),
    "best_code": str(BEST_CODE),
    "sharpe_plot": str(SHARPE_PLOT),
    "equity_plot": str(EQUITY_PLOT),
    "summary": str(SUMMARY_MD),
    "memo": str(MEMO_FILE),
    "runtime_metadata": str(RUNTIME_METADATA_FILE),
    "run_manifest": str(RUN_MANIFEST_FILE),
}

def _json_safe(value):
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, dict):
        return {str(k): _json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [_json_safe(v) for v in value]
    if isinstance(value, (np.integer, np.floating)):
        return float(value)
    return value

def sync_runtime_metadata(**updates):
    if updates:
        RUNTIME_STATE.update(updates)
    payload = dict(RUNTIME_STATE)
    payload["artifact_paths"] = dict(ARTIFACT_PATHS)
    payload["timestamp"] = time.strftime("%Y-%m-%d %H:%M:%S")
    text = json.dumps(_json_safe(payload), indent=2)
    RUNTIME_METADATA_FILE.write_text(text)
    LATEST_RUNTIME_METADATA_FILE.write_text(text)
    manifest = {
        "run_id": RUN_ID,
        "report_scope": REPORT_SCOPE,
        "summary_file": str(SUMMARY_MD),
        "runtime_metadata_file": str(RUNTIME_METADATA_FILE),
        "research_log_file": str(RESEARCH_LOG),
    }
    manifest_text = json.dumps(manifest, indent=2)
    RUN_MANIFEST_FILE.write_text(manifest_text)
    LATEST_RUN_MANIFEST_FILE.write_text(manifest_text)
    return payload

def update_runtime_state(**updates):
    return sync_runtime_metadata(**updates)

def runtime_stage_summary():
    return (
        f"llm_enabled={RUN_LLM_STAGE} | moe_enabled={RUN_MOE_STAGE} | "
        f"benchmark_mode={BENCHMARK_MODE} | reflection_enabled={REFLECTION_ENABLED}"
    )

def runtime_family_scope_summary():
    return (
        f"scope={ACTIVE_EXECUTION_SCOPE} | "
        f"executed={','.join(EXECUTED_METHOD_FAMILIES)} | "
        f"deferred={','.join(DEFERRED_METHOD_FAMILIES)}"
    )

print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"  GPU {i}: {name}  {mem:.1f} GB")
print(token_presence_text("HF_TOKEN", HF_TOKEN, HF_TOKEN_SOURCE))
print("runtime profile:", RUN_PROFILE, "| report scope:", REPORT_SCOPE)
print("configured model:", CONFIGURED_MODEL_ID,
      "| fallback:", CONFIGURED_FALLBACK_MODEL_ID)
print("stage flags:", runtime_stage_summary())
print("roadmap families:", ", ".join(ROADMAP_METHOD_FAMILIES))
print("family scope:", runtime_family_scope_summary())
sync_runtime_metadata()


## Cell 3 â€” Download and cache price data

~100 liquid US equities, 2015â€“2024 via yfinance (free).
Tickers with <70 % data coverage are dropped automatically.
Cached to parquet after first download.
Split: 2015â€“2022 = train, 2023â€“2024 = held-out test.


In [ ]:
TICKERS = [
    # â”€â”€ Tech / Semis â”€â”€
    "AAPL","MSFT","GOOGL","AMZN","META","NVDA","TSLA","AVGO",
    "ORCL","CRM","ADBE","CSCO","ACN","INTC","AMD","QCOM",
    "TXN","AMAT","MU","LRCX","ADI","IBM","INTU",
    # â”€â”€ Financials â”€â”€
    "JPM","V","MA","BAC","WFC","GS","MS","BLK","AXP","C",
    "SPGI","CME","SCHW",
    # â”€â”€ Healthcare â”€â”€
    "UNH","JNJ","LLY","PFE","ABT","TMO","MRK","ABBV",
    "DHR","AMGN","MDT","BMY","GILD","ISRG","VRTX","REGN",
    # â”€â”€ Consumer â”€â”€
    "WMT","PG","KO","PEP","COST","MCD","NKE","SBUX",
    "TGT","CL","MDLZ","EL","PM","MO",
    # â”€â”€ Industrials â”€â”€
    "CAT","HON","UPS","BA","GE","RTX","LMT","DE",
    "MMM","UNP","FDX","EMR","ETN",
    # â”€â”€ Energy â”€â”€
    "XOM","CVX","COP","SLB","EOG","PSX",
    # â”€â”€ Communication â”€â”€
    "DIS","NFLX","CMCSA","T","VZ",
    # â”€â”€ Materials â”€â”€
    "LIN","APD","ECL","SHW","FCX","NEM",
    # â”€â”€ Utilities â”€â”€
    "NEE","DUK","SO","D",
    # â”€â”€ Real Estate â”€â”€
    "PLD","AMT","CCI","SPG","EQIX",
    # â”€â”€ Retail / Other â”€â”€
    "HD","LOW","TJX","ROST","PYPL",
]

START = "2015-01-01"
END   = "2024-12-31"

def load_prices():
    if PRICES_CACHE.exists():
        df = pd.read_parquet(PRICES_CACHE)
        cols = df.columns.get_level_values(1).unique()
        print(f"cached: {len(cols)} tickers, "
              f"{df.index.min().date()} â†’ {df.index.max().date()}")
        return df
    print(f"downloading {len(TICKERS)} tickers from yfinance ...")
    last_err = None
    for attempt in range(3):
        try:
            raw = yf.download(TICKERS, start=START, end=END,
                              auto_adjust=True, progress=True, threads=True)
            if raw.empty:
                raise ValueError("empty dataframe")
            break
        except Exception as e:
            last_err = e
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(5)
    else:
        raise RuntimeError(f"yfinance download failed after 3 attempts: {last_err}")

    close = raw["Close"].dropna(how="all")

    # keep tickers with >70 % data coverage
    coverage = close.notna().mean()
    good = sorted(coverage[coverage > 0.7].index.tolist())
    close = close[good].ffill().bfill()
    # drop any remaining all-NaN columns
    close = close.dropna(axis=1, how="any")
    volume = raw["Volume"].reindex(columns=close.columns,
                                    index=close.index).fillna(0)
    df = pd.concat({"close": close, "volume": volume}, axis=1)
    df.to_parquet(PRICES_CACHE)
    print(f"saved: {close.shape[1]} tickers, {close.shape[0]} days")
    return df

prices   = load_prices()
close_all  = prices["close"].astype(float)
volume_all = prices["volume"].astype(float)

mask = close_all.index <= pd.Timestamp(TRAIN_END)
close_train,  close_test  = close_all[mask], close_all[~mask]
volume_train, volume_test = volume_all[mask], volume_all[~mask]

print(f"train : {close_train.shape}  ({close_train.index.min().date()} â†’ "
      f"{close_train.index.max().date()})")
print(f"test  : {close_test.shape}   ({close_test.index.min().date()} â†’ "
      f"{close_test.index.max().date()})")
print(f"universe: {len(close_all.columns)} tickers")

# Regime data: VIX + 10Y yield aligned to the equity trading calendar.
REGIME_CACHE = OUT / "regime.parquet"

def load_regime_data(index):
    if REGIME_CACHE.exists():
        df = pd.read_parquet(REGIME_CACHE)
        print(f"regime cached: {df.index.min().date()} -> {df.index.max().date()}")
        return df.reindex(index).ffill().bfill()
    print("downloading ^VIX and ^TNX ...")
    frames = {}
    for sym, col in [("^VIX", "VIX"), ("^TNX", "TNX")]:
        raw = yf.download(sym, start=START, end=END,
                          auto_adjust=True, progress=False)
        close_col = raw["Close"] if "Close" in raw.columns else raw.iloc[:, 0]
        frames[col] = close_col.squeeze()
    df = pd.DataFrame(frames).reindex(index).ffill().bfill()
    df.to_parquet(REGIME_CACHE)
    print(f"saved regime: {df.shape}")
    return df

regime_all = load_regime_data(close_all.index)
vix_all = regime_all["VIX"]
tnx_all = regime_all["TNX"]
vix_train, vix_test = vix_all[mask], vix_all[~mask]
tnx_train, tnx_test = tnx_all[mask], tnx_all[~mask]
print(f"VIX  train mean={vix_train.mean():.1f}  test mean={vix_test.mean():.1f}")
print(f"TNX  train mean={tnx_train.mean():.2f}%  test mean={tnx_test.mean():.2f}%")


## Cell 4 â€” Backtester with annual consistency

Vectorised pandas. Positions = `signal.shift(1)` (no lookahead).
Equal-weight across assets. 5 bps per-turn transaction cost.
Reports the primary objective Sharpe plus annual consistency.


In [ ]:
def backtest(signal_df, close_df, cost_bps=5.0):
    sig = (signal_df
           .reindex(close_df.index)
           .reindex(columns=close_df.columns)
           .astype(float).fillna(0).clip(-1, 1))
    pos = sig.shift(1).fillna(0)

    ret      = close_df.pct_change().fillna(0)
    gross    = (pos * ret).mean(axis=1)
    turnover = pos.diff().abs().mean(axis=1).fillna(0)
    cost     = turnover * (cost_bps / 10_000)
    net      = gross - cost

    bench   = ret.mean(axis=1)          # equal-weight long-only diagnostic
    active  = net - bench
    primary = net if OBJECTIVE_MODE == "market_neutral" else active

    eq         = (1 + net).cumprod()
    eq_active  = (1 + active).cumprod()
    eq_primary = (1 + primary).cumprod()

    ann_r   = net.mean()     * 252; ann_v   = net.std()     * np.sqrt(252)
    ann_ra  = active.mean()  * 252; ann_va  = active.std()  * np.sqrt(252)
    ann_rp  = primary.mean() * 252; ann_vp  = primary.std() * np.sqrt(252)
    sharpe      = float(ann_r  / ann_v)  if ann_v  > 0 else 0.0
    sharpe_act  = float(ann_ra / ann_va) if ann_va > 0 else 0.0
    sharpe_main = float(ann_rp / ann_vp) if ann_vp > 0 else 0.0

    cov  = np.cov(net.values, bench.values)
    beta = float(cov[0, 1] / cov[1, 1]) if cov[1, 1] > 0 else 0.0

    # â”€â”€ annual consistency â”€â”€
    annual_sh = []
    for yr in sorted(primary.index.year.unique()):
        a = primary[primary.index.year == yr]
        if len(a) < 50:
            continue
        v = a.std() * np.sqrt(252)
        annual_sh.append(float(a.mean() * 252 / v) if v > 0 else 0.0)
    consistency = (sum(1 for s in annual_sh if s > 0)
                   / max(len(annual_sh), 1))

    return {
        "sharpe_active": sharpe_main,
        "sharpe_raw":    sharpe,
        "benchmark_spread_sharpe": sharpe_act,
        "ann_active":    float(ann_rp),
        "ann_return":    float(ann_r),
        "ann_benchmark_spread": float(ann_ra),
        "beta":          beta,
        "max_dd":        float((eq / eq.cummax() - 1).min()),
        "max_dd_active": float((eq_primary / eq_primary.cummax() - 1).min()),
        "max_dd_benchmark_spread": float((eq_active / eq_active.cummax() - 1).min()),
        "hit_rate":      float((net > 0).mean()),
        "avg_turnover":  float(turnover.mean()),
        "equity":        eq,
        "equity_active": eq_active,
        "equity_primary": eq_primary,
        "annual_sharpes": annual_sh,
        "consistency":    consistency,
        "min_annual":     min(annual_sh) if annual_sh else 0.0,
        "objective_mode": OBJECTIVE_MODE,
        "primary_objective_label": PRIMARY_OBJECTIVE_LABEL,
    }

# â”€â”€ sanity check: 20-day cross-sectional momentum â”€â”€
_cs = close_train.pct_change(20).rank(axis=1, pct=True) * 2 - 1
_cs = _cs.sub(_cs.mean(axis=1), axis=0)
_m  = backtest(_cs, close_train)
print(f"sanity: mode={OBJECTIVE_MODE} primarySh={_m['sharpe_active']:+.2f}  benchSpread={_m['benchmark_spread_sharpe']:+.2f}  "
      f"beta={_m['beta']:+.2f}  consistency={_m['consistency']:.0%}  "
      f"DD={_m['max_dd_active']:+.1%}")


## Cell 5 â€” Sandbox executor

AST-validated, restricted builtins, thread-based timeout.
Works from any thread (main or ThreadPoolExecutor worker).


In [ ]:
FORBIDDEN = [
    "import os", "import sys", "import subprocess", "import socket",
    "import shutil", "import requests", "import urllib",
    "open(", "__import__", "eval(", "exec(",
    "compile(", "globals(", "locals(", "getattr(", "setattr(", "delattr(",
    "input(", "quit(", "exit(", "__builtins__", "__class__", "__bases__",
    "__subclasses__", "pathlib", "Path(",
]
ALLOWED_IMPORTS = {"numpy", "np", "pandas", "pd", "math"}

def validate_code(code_str):
    low = code_str.lower()
    for bad in FORBIDDEN:
        if bad.lower() in low:
            return False, f"forbidden: {bad!r}"
    try:
        tree = ast.parse(code_str)
    except SyntaxError as e:
        return False, f"syntax: {e}"
    for node in ast.walk(tree):
        if isinstance(node, ast.Import):
            for alias in node.names:
                if alias.name.split(".")[0] not in ALLOWED_IMPORTS:
                    return False, f"disallowed import: {alias.name}"
        elif isinstance(node, ast.ImportFrom):
            if (node.module or "").split(".")[0] not in ALLOWED_IMPORTS:
                return False, f"disallowed from-import: {node.module}"
    # Cheap static lint for row-Series/DataFrame broadcast bugs that repeatedly
    # create shape explosions in generated code.
    if re.search(r"\.mean\s*\(\s*axis\s*=\s*1\s*\)\s*\.reindex_like\s*\(", code_str):
        return False, "shape-risk: row-wise Series reindex_like(DataFrame); use .sub(..., axis=0) or .values[:, None]"
    if re.search(r"\.mean\s*\(\s*axis\s*=\s*1\s*\)\s*\.rank\s*\(\s*axis\s*=\s*1", code_str):
        return False, "series-axis-risk: axis=1 rank called on row-wise Series"
    if re.search(r"\.std\s*\(\s*axis\s*=\s*1\s*\)\s*\.rank\s*\(\s*axis\s*=\s*1", code_str):
        return False, "series-axis-risk: axis=1 rank called on row-wise Series"
    if re.search(r"pd\.concat\s*\(", code_str) or re.search(r"\.join\s*\(", code_str) or re.search(r"\.merge\s*\(", code_str):
        return False, "shape-risk: concat/join/merge inside signal is disallowed"
    if re.search(r"\.stack\s*\(", code_str) or re.search(r"\.unstack\s*\(", code_str) or re.search(r"\.pivot\s*\(", code_str) or re.search(r"\.melt\s*\(", code_str):
        return False, "shape-risk: stack/unstack/pivot/melt inside signal is disallowed"
    if "def signal" not in code_str:
        return False, "no `def signal` found"
    return True, "ok"

SAFE_BUILTINS = {
    k: (__builtins__[k] if isinstance(__builtins__, dict)
        else getattr(__builtins__, k))
    for k in ["len","range","min","max","abs","sum","int","float","bool",
              "list","dict","tuple","set","str","True","False","None",
              "enumerate","zip","sorted","reversed","map","filter","round",
              "isinstance","type","any","all","print","slice"]
    if (k in __builtins__ if isinstance(__builtins__, dict)
        else hasattr(__builtins__, k))
}

def run_signal_code(code_str, close_df, volume_df, vix_s=None, tnx_s=None, timeout=SANDBOX_TIMEOUT):
    """Execute signal code in a sandboxed thread with timeout."""
    ok, msg = validate_code(code_str)
    if not ok:
        return None, f"VALIDATION: {msg}"

    ns = {"np": np, "pd": pd, "__builtins__": SAFE_BUILTINS}
    result = [None, None]          # [output, error_string]

    def _run():
        try:
            exec(code_str, ns)
            if "signal" not in ns or not callable(ns["signal"]):
                result[1] = "no callable `signal` after exec"
                return
            try:
                params = set(_inspect.signature(ns["signal"]).parameters)
            except (ValueError, TypeError):
                params = set()
            kwargs = {}
            if "vix" in params:
                kwargs["vix"] = vix_s.copy() if vix_s is not None else None
            if "tnx" in params:
                kwargs["tnx"] = tnx_s.copy() if tnx_s is not None else None
            result[0] = ns["signal"](close_df.copy(), volume_df.copy(), **kwargs)
        except Exception as e:
            result[1] = f"{type(e).__name__}: {e}"

    t = _th.Thread(target=_run, daemon=True)
    t.start()
    t.join(timeout=timeout)
    if t.is_alive():
        return None, f"TIMEOUT (>{timeout}s)"
    if result[1] is not None:
        return None, result[1]
    out = result[0]

    if not isinstance(out, pd.DataFrame):
        return None, f"returned {type(out).__name__}, need DataFrame"
    if out.shape != close_df.shape:
        return None, f"shape {out.shape} != expected {close_df.shape}"
    if out.columns.duplicated().any():
        return None, "SIGNAL_VALIDATION: duplicated columns"
    if not out.index.equals(close_df.index):
        return None, "SIGNAL_VALIDATION: index mismatch vs close"
    if not out.columns.equals(close_df.columns):
        return None, "SIGNAL_VALIDATION: column mismatch vs close"
    ok_signal, sig_msg = validate_signal(out, expected_shape=close_df.shape)
    if not ok_signal:
        return None, f"SIGNAL_VALIDATION: {sig_msg}"
    out = out.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-1, 1)
    return out, None

def validate_signal(sig_df, expected_shape=None):
    """Reject structurally invalid or degenerate signals."""
    s = sig_df.astype(float).replace([np.inf, -np.inf], np.nan).fillna(0.0).clip(-1, 1)
    if expected_shape is not None and s.shape != expected_shape:
        return False, f"shape {s.shape} != expected {expected_shape}"
    if s.columns.duplicated().any():
        return False, "exploded column set"
    if s.std().mean() < 0.03:
        return False, "near-constant signal (std<0.03)"
    long_frac  = float((s >  0.05).mean().mean())
    short_frac = float((s < -0.05).mean().mean())
    if min(long_frac, short_frac) < 0.05:
        return False, f"not long-short (L={long_frac:.2f} S={short_frac:.2f})"
    if s.mean().abs().mean() > 0.5:
        return False, f"directionally biased (|mean|={s.mean().abs().mean():.2f})"
    return True, "ok"

def detect_lookahead(code_str, close_df, volume_df, vix_s=None, tnx_s=None, split_frac=0.6, seed=0):
    """Shuffle future data and check if past-slice signal changes."""
    sig_real, err = run_signal_code(code_str, close_df, volume_df, vix_s=vix_s, tnx_s=tnx_s, timeout=20)
    if err is not None:
        return False, err
    T = int(len(close_df) * split_frac)
    rng = np.random.RandomState(seed)
    perm = rng.permutation(len(close_df) - T)
    c2, v2 = close_df.copy(), volume_df.copy()
    vix2 = None
    tnx2 = None
    if vix_s is not None:
        vix2 = vix_s.copy(); vix2.iloc[T:] = vix_s.iloc[T:].values[perm]
    if tnx_s is not None:
        tnx2 = tnx_s.copy(); tnx2.iloc[T:] = tnx_s.iloc[T:].values[perm]
    c2.iloc[T:] = close_df.iloc[T:].values[perm]
    v2.iloc[T:] = volume_df.iloc[T:].values[perm]
    sig_perm, err = run_signal_code(code_str, c2, v2, vix_s=vix2, tnx_s=tnx2, timeout=20)
    if err is not None:
        return False, f"shuffle_run: {err}"
    a = sig_real.iloc[:T].fillna(0).values
    b = sig_perm.iloc[:T].fillna(0).values
    diff = float(np.abs(a - b).mean())
    if diff > 1e-6:
        return False, f"LOOKAHEAD detected (diff={diff:.6f})"
    return True, "ok"

# â”€â”€ self-test â”€â”€
_test = """
def signal(close, volume):
    r = close.pct_change(20).rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    return out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
"""
_s, _e = run_signal_code(_test, close_train, volume_train)
print("sandbox self-test:", "OK" if _e is None else f"FAIL: {_e}")


# â”€â”€ Remediation V3: Executable Panel Precheck Helper â”€â”€
def precheck_candidate(code_str, close_df, volume_df, vix_s=None, tnx_s=None, timeout=30):
    """
    Lightweight pre-backtest validation.
    Returns a stable dict with failure_class for repair prompting.
    """
    result = {
        "is_executable": False,
        "failure_class": None,
        "failure_detail": None,
        "output_type": None,
        "output_shape": None,
        "index_match": False,
        "columns_match": False,
        "finite_ok": False,
        "activity_stats": {
            "panel_std": None,
            "cross_sectional_dispersion": None,
            "mean_abs_exposure": None,
        },
    }

    try:
        sig, err = run_signal_code(
            code_str, close_df, volume_df,
            vix_s=vix_s, tnx_s=tnx_s, timeout=timeout
        )
    except Exception as e:
        result["failure_class"] = "runtime_error"
        result["failure_detail"] = str(e)[:300]
        return result

    if err:
        # Classify common errors
        err_lower = str(err).lower()
        if "shape" in err_lower or "broadcast" in err_lower:
            result["failure_class"] = "shape_mismatch"
        elif "boolean" in err_lower and ("subtract" in err_lower or "arithmetic" in err_lower):
            result["failure_class"] = "boolean_arithmetic"
        else:
            result["failure_class"] = "runtime_error"
        result["failure_detail"] = str(err)[:300]
        return result

    if not isinstance(sig, pd.DataFrame):
        result["failure_class"] = "type_mismatch"
        result["failure_detail"] = f"Expected DataFrame, got {type(sig)}"
        result["output_type"] = str(type(sig))
        return result

    result["output_type"] = "DataFrame"
    result["output_shape"] = sig.shape
    result["index_match"] = list(sig.index) == list(close_df.index)
    result["columns_match"] = list(sig.columns) == list(close_df.columns)

    if not (result["index_match"] and result["columns_match"]):
        result["failure_class"] = "shape_mismatch"
        result["failure_detail"] = f"Shape mismatch: got {sig.shape} vs expected {close_df.shape}"
        return result

    # Finite check
    if not np.isfinite(sig.values).all():
        result["failure_class"] = "non_finite"
        result["failure_detail"] = "Non-finite values present"
        return result
    result["finite_ok"] = True

    # Activity / degeneracy checks
    panel_std = float(sig.values.std())
    cross_disp = float(sig.std(axis=1).mean())
    mean_abs = float(np.abs(sig.values).mean())

    result["activity_stats"] = {
        "panel_std": panel_std,
        "cross_sectional_dispersion": cross_disp,
        "mean_abs_exposure": mean_abs,
    }

    if panel_std < STAGE_B_MIN_ACTIVITY_STD or cross_disp < STAGE_B_MIN_CROSS_SECTIONAL_DISPERSION:
        result["failure_class"] = "degenerate"
        result["failure_detail"] = f"Low activity: std={panel_std:.4f}, disp={cross_disp:.4f}"
        return result

    # Simple directional bias check
    mean_sign = np.sign(sig.values).mean()
    if abs(mean_sign) > STAGE_B_MAX_DIRECTIONAL_BIAS:
        result["failure_class"] = "directional_bias"
        result["failure_detail"] = f"Strong directional bias: {mean_sign:.2f}"
        return result

    result["is_executable"] = True
    return result



## Cell 6 â€” Load model (4-bit quantised)

The model load is lazy. If no LLM stage is enabled, this cell skips the
bitsandbytes/CUDA path entirely so deterministic research can still run.


In [ ]:
REQUESTED_LLM_MODEL = bool(RUN_LLM_STAGE or RUN_MOE_STAGE or RUN_LLM_SMOKE)
SHOULD_LOAD_LLM_MODEL = bool(REQUESTED_LLM_MODEL and RUN_BNB_MODEL_LOAD)
tok = None
model = None
update_runtime_state(
    llm_stage_enabled=bool(RUN_LLM_STAGE),
    moe_stage_enabled=bool(RUN_MOE_STAGE),
    llm_smoke_enabled=bool(RUN_LLM_SMOKE),
    bnb_model_load_enabled=bool(RUN_BNB_MODEL_LOAD),
    llm_stage_loaded=False,
    llm_stage_executed=False,
    actual_model_id=None,
)

def _record_cuda_inventory():
    n_gpus = torch.cuda.device_count()
    names = [torch.cuda.get_device_name(i) for i in range(n_gpus)]
    update_runtime_state(
        cuda_device_count=n_gpus,
        cuda_device_names=names,
    )
    return names

if SHOULD_LOAD_LLM_MODEL:
    _record_cuda_inventory()

hub_kwargs = {"token": HF_TOKEN} if HF_TOKEN else {}

def _load_tokenizer(model_id):
    t = AutoTokenizer.from_pretrained(model_id, **hub_kwargs)
    if t.pad_token_id is None:
        t.pad_token_id = t.eos_token_id
    return t

def _bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

def _load_model(model_id):
    return AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=_bnb_config(),
        device_map="auto",
        dtype=torch.float16,
        **hub_kwargs,
    )

def ensure_model_loaded():
    global tok, model, MODEL_ID, ACTIVE_MODEL_ID
    if tok is not None and model is not None:
        return tok, model
    if not RUN_BNB_MODEL_LOAD:
        update_runtime_state(llm_stage_error="bnb_model_load_disabled")
        raise RuntimeError(
            "LLM model loading is disabled for this Kaggle run because the current "
            "CUDA/bitsandbytes stack can hard-crash the kernel. Set "
            "AUTORESEARCH_ENABLE_BNB_LOAD=1 only for a dedicated model-load test."
        )

    active_model_id = CONFIGURED_MODEL_ID
    load_strategy = "configured_primary"
    tok = _load_tokenizer(active_model_id)
    try:
        model = _load_model(active_model_id)
    except Exception as e:
        msg = str(e).lower()
        is_oom = ("outofmemory" in msg) or ("out of memory" in msg) or isinstance(e, torch.OutOfMemoryError)
        if (not is_oom) or active_model_id == FALLBACK_MODEL_ID:
            update_runtime_state(llm_stage_error=f"{type(e).__name__}: {e}")
            raise
        print(f"Primary model OOM: {CONFIGURED_MODEL_ID}")
        print("Falling back to configured backup model for stability on current T4 session...")
        gc.collect(); torch.cuda.empty_cache()
        active_model_id = CONFIGURED_FALLBACK_MODEL_ID
        load_strategy = "oom_fallback"
        tok = _load_tokenizer(active_model_id)
        model = _load_model(active_model_id)

    model.eval()
    ACTIVE_MODEL_ID = active_model_id
    MODEL_ID = active_model_id
    update_runtime_state(
        actual_model_id=ACTIVE_MODEL_ID,
        llm_stage_loaded=True,
        llm_stage_error=None,
        llm_load_strategy=load_strategy,
    )
    print(f"model loaded: configured={CONFIGURED_MODEL_ID} actual={MODEL_ID} ({load_strategy})")
    vram = torch.cuda.memory_allocated() / 1e9
    print(f"VRAM allocated: {vram:.1f} GB")
    return tok, model

if SHOULD_LOAD_LLM_MODEL:
    ensure_model_loaded()
    if RUN_LLM_SMOKE:
        try:
            smoke_ids = tok("LLM smoke test", return_tensors="pt").input_ids.to(model.device)
            with torch.no_grad():
                _ = model.generate(smoke_ids, max_new_tokens=1, do_sample=False, pad_token_id=tok.pad_token_id)
            update_runtime_state(llm_smoke_executed=True, llm_smoke_ok=True)
            print("LLM smoke test: OK")
        except Exception as e:
            update_runtime_state(llm_smoke_executed=True, llm_smoke_ok=False, llm_smoke_error=f"{type(e).__name__}: {e}")
            print(f"LLM smoke test failed: {type(e).__name__}: {e}")
else:
    if REQUESTED_LLM_MODEL:
        update_runtime_state(llm_load_strategy="skipped_bnb_disabled", llm_stage_error="bnb_model_load_disabled")
        print("LLM stage requested, but bitsandbytes model load is disabled for kernel stability.")
        print("Set AUTORESEARCH_ENABLE_BNB_LOAD=1 only for a dedicated model-load smoke test.")
    else:
        update_runtime_state(llm_load_strategy="skipped_disabled")
        print("LLM stages disabled; skipping model load.")


## Cell 7 â€” LLM generation helper

In [ ]:
@torch.inference_mode()
def llm(messages, max_new_tokens=LLM_BATCH_MAX_NEW_TOKENS, temperature=0.8, top_p=0.95):
    ensure_model_loaded()
    update_runtime_state(
        llm_stage_executed=True,
        llm_calls=int(RUNTIME_STATE.get("llm_calls", 0)) + 1,
        actual_model_id=ACTIVE_MODEL_ID or MODEL_ID,
    )
    prompt = tok.apply_chat_template(messages, tokenize=False,
                                      add_generation_prompt=True)
    enc = tok(prompt, return_tensors="pt", truncation=True,
              max_length=LLM_PROMPT_MAX_TOKENS).to(model.device)
    try:
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-5),
            top_p=top_p,
            pad_token_id=tok.pad_token_id,
        )
    finally:
        pass
    gen = out[0, enc["input_ids"].shape[1]:]
    text = tok.decode(gen, skip_special_tokens=True).strip()
    del gen, out, enc
    gc.collect(); torch.cuda.empty_cache()
    return text


## Cell 8 â€” Prompts + candidate extraction

The system prompt includes a **feature catalog** so the model knows what
building blocks are available. The user prompt includes the research log,
the latest reflection memo, and recent failure examples.


In [ ]:
SYSTEM_PROMPT = """You are a quantitative researcher discovering alpha signals for a long-short equity strategy on ~100 US stocks.

Given:
- `close`: pd.DataFrame of daily adjusted close prices (rows=dates, cols=tickers)
- `volume`: pd.DataFrame of daily volumes (same shape)

Write: def signal(close, volume, vix=None, tnx=None) -> pd.DataFrame
Return values in [-1, +1]. +1 = strongest long, -1 = strongest short.
Positions are lagged by 1 day automatically â€” no lookahead needed in your code.

Hard rules:
- Use ONLY numpy (np) and pandas (pd). No other imports.
- Return shape MUST equal close.shape.
- Keep functions under 35 lines.
- End EVERY signal with:
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)

CRITICAL â€” lookahead bias auto-rejects your signal:
- NEVER use full-series aggregations: close.mean(), volume.std(), close.quantile()
  These use FUTURE data. They will be detected and rejected.
- ONLY use trailing windows: .pct_change(N), .rolling(N).mean(), .rolling(N).std(),
  .ewm(span=N).mean(), .shift(N) with N > 0.
- Cross-sectional (axis=1) ops on a single date are fine:
  .rank(axis=1), .mean(axis=1), .std(axis=1)

CRITICAL â€” signal must be long-short:
- Both positive AND negative values on most days.
- A near-constant signal will be rejected.
- Pattern: r = ...rank(axis=1, pct=True); out = (r - 0.5) * 2

Feature catalog (all require explicit lookback N):
  Momentum:       close.pct_change(N)
  Ranking:        .rank(axis=1, pct=True) â†’ cross-sectional percentile [0,1]
  Volatility:     close.pct_change().rolling(N).std()
  MA ratio:       close / close.rolling(N).mean() - 1
  EWM crossover:  close.ewm(span=N).mean() / close.ewm(span=M).mean() - 1
  Volume ratio:   volume / volume.rolling(N).mean()
  Bollinger z:    (close - close.rolling(N).mean()) / close.rolling(N).std()
  12-1 momentum:  close.pct_change(252) - close.pct_change(21)
  Multi-horizon:  average of rank signals at different lookbacks


# â”€â”€ Remediation V3 Hardening (added from May 27 2026 failure analysis) â”€â”€
# CRITICAL OUTPUT CONTRACT:
# - Your function MUST return a pandas DataFrame with EXACTLY the same
#   index as `close` and EXACTLY the same columns as `close`.
# - VIX and TNX (if provided) are 1-dimensional Series. You MUST broadcast
#   them explicitly across columns, e.g.:
#       vix_panel = vix.values[:, None]   # or pd.DataFrame
# - Never do arithmetic directly on boolean Series from regime conditions.
#   Always cast to float first:
#       regime = (vix.rolling(5).mean() < 25).astype(float).values[:, None]
#
# Common failures observed in May 2026 runs (AVOID THESE):
# 1. Shape explosion: doing .rank(axis=1) or operations that expand dimensions.
# 2. Boolean subtraction: (close > ma) - (close < ma)  â†’ TypeError
# 3. Forgetting to broadcast 1-D regime inputs to the panel.
#
# GOOD minimal example (momentum with VIX regime):
# def signal(close, volume, vix=None, tnx=None):
#     mom = close.pct_change(63).rank(axis=1, pct=True) * 2 - 1
#     if vix is not None:
#         regime = (vix.rolling(5).mean() < 25).astype(float).values[:, None]
#         mom = mom * regime
#     out = mom.sub(mom.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
#     return out
"""

USER_PROMPT_TMPL = """## Research log (sorted by active Sharpe)
{log_summary}

## Research memo
{memo}
{failures}

---
Batch {iter_n} of {n_batches}.

Propose 3 NEW signal functions, each testing a DIFFERENT hypothesis.
Vary across: lookback horizons, feature combinations, normalization, filters.
Do NOT repeat signals that already appear in the research log.

Output format â€” follow EXACTLY:

=== CANDIDATE 1 ===
HYPOTHESIS: <one sentence>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return result
```

=== CANDIDATE 2 ===
HYPOTHESIS: <one sentence>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return result
```

=== CANDIDATE 3 ===
HYPOTHESIS: <one sentence>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return result
```
"""

REFLECTION_PROMPT = """Review this experiment log and write a research memo.

## Successful signals (sorted by active Sharpe)
{successes}

## Failures
{failure_summary}

## Current best
{best_info}

Write a concise RESEARCH MEMO (5-7 bullet points):
1. Which strategy families show the most promise? Cite Sharpe values.
2. What lookback horizons work best?
3. What is the most common failure mode?
4. What areas of the strategy space are UNEXPLORED?
5. What specific experiments should be tried next?
6. Could any existing signals be combined?
Be specific and quantitative.
"""

# â”€â”€ Extraction â”€â”€
_CAND_RE = re.compile(
    r"===\s*CANDIDATE\s*(\d+)\s*===(.*?)(?====\s*CANDIDATE|\Z)",
    re.DOTALL | re.IGNORECASE)
_CODE_RE = re.compile(r"```(?:python)?\s*\n(.*?)```", re.DOTALL)
_HYP_RE  = re.compile(r"HYPOTHESIS:\s*(.+?)(?:\n|$)", re.IGNORECASE)

def extract_candidates(text):
    """Extract up to 3 candidates from LLM output."""
    results = []
    for m in _CAND_RE.finditer(text):
        body = m.group(2)
        cm = _CODE_RE.search(body)
        if not cm:
            continue
        hm = _HYP_RE.search(body)
        results.append({
            "idx": int(m.group(1)),
            "hypothesis": hm.group(1).strip() if hm else "",
            "code": cm.group(1).strip(),
        })
    # fallback: just extract all code blocks
    if not results:
        for i, cm in enumerate(_CODE_RE.finditer(text)):
            code_text = cm.group(1).strip()
            if "def signal" in code_text:
                results.append({"idx": i + 1, "hypothesis": "", "code": code_text})
    return results[:3]

# -- Adapted prompt pack for this notebook's simpler loop --
SYSTEM_PROMPT = f"""You are a quantitative researcher discovering alpha signals for a market-neutral long-short equity strategy on ~100 US stocks.

INPUTS
- `close`: pd.DataFrame of daily adjusted close prices (rows=dates, cols=tickers)
- `volume`: pd.DataFrame of daily volumes (same shape)

Write: def signal(close, volume, vix=None, tnx=None) -> pd.DataFrame
Return values in [-1, +1]. +1 = strongest long, -1 = strongest short.
Positions are lagged by 1 day automatically in the harness. Do not add your own forward-looking shift.

WHAT YOU ARE REALLY OPTIMIZING
The notebook uses a train-time research score based on metrics it actually measures:
- {PRIMARY_OBJECTIVE_LABEL} is the main objective
- `vix`, `tnx`: optional pd.Series regime inputs aligned to the equity dates
- consistency (fraction of positive calendar-year Sharpes) is a bonus
- min_annual matters because unstable signals often fail held-out review
- beta above 0.10 is penalized
- turnover above 0.50 is penalized
- very deep drawdowns are penalized
- benchmark-relative Sharpe is a diagnostic, not the main target in market-neutral mode

The fastest way to lose is to maximize train Sharpe while ignoring stability.
Prefer signals that should stay long-short, diversified, and time-stable.

HARD RULES
1. Use ONLY numpy (np) and pandas (pd). No imports.
2. Return shape MUST equal close.shape.
3. Keep functions under 40 lines.
4. End EVERY signal with:
       out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
5. Do NOT use one-sided threshold rules that collapse to all-long or all-short.
6. Prefer percentile ranks:
       r = feature.rank(axis=1, pct=True)
       out = (r - 0.5) * 2
7. If you compute a row-wise Series such as feature.mean(axis=1), broadcast it back with
       feature.sub(row_series, axis=0)
   or
       feature - row_series.values[:, None]
   Never use row_series.reindex_like(feature) inside arithmetic.
8. Never use pd.concat, join, merge, stack, unstack, pivot, melt, transpose, or reset_index inside signal().
9. Never call axis=1 methods on a Series. If an expression reduces to a Series, broadcast it back immediately.
10. Return exactly the same index and columns as close. Do not create new columns, reorder columns, or widen the matrix.
11. Bad examples that will be rejected:
    - feature.mean(axis=1).rank(axis=1)
    - pd.concat([...], axis=1) inside signal()
    - any output with shape like (dates, dates*tickers) or columns not equal to close.columns

LOOKAHEAD BIAS = AUTO-REJECT
- NEVER use full-series aggregations: close.mean(), volume.std(), close.quantile()
- ONLY use trailing windows: .pct_change(N), .rolling(N).mean(), .rolling(N).std(),
  .ewm(span=N).mean(), .shift(N) with N > 0
- Cross-sectional same-date ops are fine: .rank(axis=1), .mean(axis=1), .std(axis=1)

SIGNAL VALIDITY
- Both positive AND negative values on most days
- Near-constant cross-sections are rejected
- Strong directional bias is rejected

FACTOR FAMILY MENU
You are NOT limited to EWM crossovers. Legitimate families include:
1. momentum      - medium-term return ranks, multi-horizon momentum, 12-1 style momentum
2. mean_reversion - short-term loser bounce, pullback after stretched moves
3. volume        - price move confirmed by relative volume, turnover-conditioned trend
4. volatility    - vol-scaled momentum, low-vol continuation, breakout normalized by vol
5. ewm           - EWM crossover, EWM spread normalized by vol, EWM with trailing filters
6. multi_factor  - blends of two or more weakly correlated ranked signals
7. regime        - switching weights based on trailing realized volatility or volume stress

Use under-explored families when the search is stagnating.

SEED TEMPLATES
- momentum seed: 20d or 63d return rank, optionally vol-scaled
- mean_reversion seed: negative 5d return rank, optionally gated by recent stretch
- ewm seed: fast/slow EWM spread normalized by trailing vol
- volume seed: 21d return multiplied by relative volume rank
- regime seed: momentum blended with reversal under trailing stress
"""

USER_PROMPT_TMPL = """## Research log (sorted by research score)
{log_summary}

## Current best candidate
{current_best}

## Deterministic seed parents under the live objective
{seed_summary}

## Family balance in the current log
{family_summary}

## Research memo
{memo}
{failures}

## Stagnation status
{stagnation_status}

---
Batch {iter_n} of {n_batches}.

INSTRUCTIONS:
{diversity_instructions}

STRUCTURE IS SCORED BEFORE ALPHA.
If a candidate is missing PARENT_ID, MUTATION_TYPE, FAMILY, HYPOTHESIS, or ROBUSTNESS_RATIONALE,
it will be discarded before backtesting.

For EACH candidate provide:
  - PARENT_ID: copy an existing iter id when refining prior work, or "seed" if starting fresh
  - MUTATION_TYPE: what structural change you are making
  - FAMILY: one of momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor
  - HYPOTHESIS: one sentence of economic rationale
  - ROBUSTNESS_RATIONALE: one sentence explaining why this should survive across years
  - code block

Hard constraints:
  - Do NOT repeat signals already in the research log.
  - Do NOT include import lines.
  - Do NOT produce all-long or all-short signals.
  - Do NOT use pd.concat, join, merge, stack, unstack, pivot, melt, transpose, or reset_index inside signal().
  - Do NOT call axis=1 on a Series; if you reduce row-wise, broadcast back with .sub(..., axis=0) or .values[:, None].
  - Return a DataFrame with EXACTLY close.index and close.columns; never create or widen columns.
  - Candidate 1 should mutate the current best template only when a viable parent exists; otherwise start from a deterministic seed parent.
  - Candidate 2 must come from a different family than candidate 1.
  - Candidate 3 must be either a blend or a regime-conditioned variant.
  - Generic 5d/10d/20d rank-only momentum and naive mislabeled reversion have already failed; do not repeat them.

Output format - follow EXACTLY:

=== CANDIDATE 1 ===
PARENT_ID: <iter id or "seed">
MUTATION_TYPE: <what changed>
FAMILY: <momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor>
HYPOTHESIS: <economic rationale in one sentence>
ROBUSTNESS_RATIONALE: <why the effect should remain stable across years>
```python
def signal(close, volume, vix=None, tnx=None):
    # raw_signal must stay close-shaped throughout
    ...
    out = (raw_signal.rank(axis=1, pct=True) - 0.5) * 2
    return out
```

=== CANDIDATE 2 ===
PARENT_ID: <iter id or "seed">
MUTATION_TYPE: <what changed>
FAMILY: <momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor>
HYPOTHESIS: <economic rationale in one sentence>
ROBUSTNESS_RATIONALE: <why the effect should remain stable across years>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```

=== CANDIDATE 3 ===
PARENT_ID: <iter id or "seed">
MUTATION_TYPE: <what changed>
FAMILY: <momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor>
HYPOTHESIS: <economic rationale in one sentence>
ROBUSTNESS_RATIONALE: <why the effect should remain stable across years>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```
"""

REFLECTION_PROMPT = """Review this experiment log and write a research memo.

## Top scored signals (sorted by research score)
{successes}

## Family / cluster summary
{cluster_summary}

## Failures
{failure_summary}

## Current best
{best_info}

## Stagnation context
Generations without improvement: {stagnation_gens}
Most common failure reason: {top_failure_reason}

Write a RESEARCH MEMO with exactly these sections:

### 1. WHAT IS WORKING
For each promising family, cite research score, primary Sharpe, consistency, and min_annual.
If all cited ideas have negative score or negative Sharpe, start this section with exactly: Nothing is working yet.

### 2. ROBUSTNESS DIAGNOSIS
For the top 3 ideas, explain why they may fail across years.
Focus on consistency, min_annual, beta drift, turnover, and drawdown behavior.

### 3. UNEXPLORED TERRITORY
List 3 structural ideas or families that have not been explored enough.
For each give:
- family name
- economic rationale
- a 3-5 line code sketch

### 4. ANTI-PATTERN LIST
List the top 3 mutation patterns wasting compute and why.

### 5. NEXT 3 HYPOTHESES
Give three specific next experiments.
Each must include family, mutation type, and robustness argument.
At least one must come from unexplored territory.

Be specific and quantitative. Do not recycle vague EWM advice.
"""

def _field(pattern, body, default=""):
    match = re.search(pattern, body, re.IGNORECASE)
    return match.group(1).strip() if match else default

VALID_FAMILIES = {
    "momentum", "mean_reversion", "volume",
    "volatility", "ewm", "regime", "multi_factor",
}

def _normalize_family(value):
    fam = (value or "").strip().lower()
    return fam if fam in VALID_FAMILIES else "unknown"

def _is_structured_candidate(cand):
    return (
        bool(cand.get("parent_id"))
        and bool(cand.get("mutation_type"))
        and _normalize_family(cand.get("family")) != "unknown"
        and bool(cand.get("hypothesis"))
        and bool(cand.get("robustness_rationale"))
        and "def signal" in (cand.get("code") or "")
    )

def code_fingerprint(code_text):
    normalized = re.sub(r"#.*", "", code_text or "")
    normalized = re.sub(r"\s+", " ", normalized).strip().lower()
    return hashlib.sha1(normalized.encode("utf-8")).hexdigest()[:16]

def validate_candidate_semantics(cand):
    family = _normalize_family(cand.get("family"))
    code = (cand.get("code") or "").lower()
    if family == "mean_reversion":
        if not any(token in code for token in ["0.5 -", "-close.pct_change", "-ret", "-returns", "(-", "negative"]):
            return False, "SEMANTIC_FAMILY: mean_reversion must invert recent-return direction"
    elif family == "volume":
        if code.count("volume") <= 1 or not any(token in code for token in ["volume.rolling", "volume.pct_change", "volume /", "/ volume"]):
            return False, "SEMANTIC_FAMILY: volume family must materially depend on volume features"
    elif family == "multi_factor":
        if code.count(".rank(axis=1") < 2:
            return False, "SEMANTIC_FAMILY: multi_factor must combine at least two ranked components"
    return True, "ok"

REPAIR_PROMPT = """Rewrite the draft response into the EXACT 3-candidate format below.
Keep the same candidate ideas when possible, but repair missing metadata and invalid formatting.
Do not add imports. Do not add prose outside the candidate blocks.

=== CANDIDATE 1 ===
PARENT_ID: <iter id or "seed">
MUTATION_TYPE: <what changed>
FAMILY: <momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor>
HYPOTHESIS: <economic rationale in one sentence>
ROBUSTNESS_RATIONALE: <why the effect should remain stable across years>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```

=== CANDIDATE 2 ===
PARENT_ID: <iter id or "seed">
MUTATION_TYPE: <what changed>
FAMILY: <momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor>
HYPOTHESIS: <economic rationale in one sentence>
ROBUSTNESS_RATIONALE: <why the effect should remain stable across years>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```

=== CANDIDATE 3 ===
PARENT_ID: <iter id or "seed">
MUTATION_TYPE: <what changed>
FAMILY: <momentum|mean_reversion|volume|volatility|ewm|regime|multi_factor>
HYPOTHESIS: <economic rationale in one sentence>
ROBUSTNESS_RATIONALE: <why the effect should remain stable across years>
```python
def signal(close, volume, vix=None, tnx=None):
    ...
    return out
```
"""

def extract_candidates(text):
    """Extract up to 3 candidates from LLM output."""
    results = []
    for m in _CAND_RE.finditer(text):
        body = m.group(2)
        cm = _CODE_RE.search(body)
        if not cm:
            continue
        results.append({
            "idx": int(m.group(1)),
            "parent_id": _field(r"PARENT_ID:\s*(.+?)(?:\n|$)", body, "seed"),
            "mutation_type": _field(r"MUTATION_TYPE:\s*(.+?)(?:\n|$)", body, "unspecified"),
            "family": _normalize_family(_field(r"FAMILY:\s*(.+?)(?:\n|$)", body, "unknown")),
            "hypothesis": _field(r"HYPOTHESIS:\s*(.+?)(?:\n|$)", body),
            "robustness_rationale": _field(r"ROBUSTNESS_RATIONALE:\s*(.+?)(?:\n|$)", body),
            "code": cm.group(1).strip(),
        })
    return [cand for cand in results[:3] if _is_structured_candidate(cand)]

def repair_and_extract_candidates(text):
    cands = extract_candidates(text)
    if len(cands) >= 3:
        return cands, text, False
    repaired = llm(
        [
            {"role": "system", "content": "You are a strict output formatter."},
            {"role": "user", "content": REPAIR_PROMPT + "\n\nDRAFT RESPONSE:\n" + text[:6000]},
        ],
        max_new_tokens=LLM_REPAIR_MAX_NEW_TOKENS,
        temperature=0.0,
        top_p=1.0,
    )
    return extract_candidates(repaired), repaired, True


## Cell 9 â€” Research log + reflection memo

Append-only JSONL log. The log is the model's **memory**: it sees its own
past experiments (ranked by Sharpe) plus recent failures. The reflection
memo is a separate file updated every few batches.


In [ ]:
def append_log(entry):
    enriched = dict(entry)
    enriched.setdefault("run_id", RUN_ID)
    enriched.setdefault("run_profile", RUN_PROFILE)
    with open(RESEARCH_LOG, "a") as f:
        f.write(json.dumps(enriched, default=str) + "\n")

def load_log():
    if not RESEARCH_LOG.exists():
        return []
    out = []
    for line in open(RESEARCH_LOG):
        line = line.strip()
        if not line:
            continue
        try:
            out.append(json.loads(line))
        except Exception:
            pass
    return out

def save_memo(text):
    MEMO_FILE.write_text(text)
    update_runtime_state(last_memo_file=str(MEMO_FILE))

def load_memo():
    if MEMO_FILE.exists():
        return MEMO_FILE.read_text()
    return "(No memo yet â€” early exploration phase.)"

def log_summary_for_prompt(log, top_k=5, code_lines=6):
    good = [e for e in log if e.get("sharpe") is not None]
    if not good:
        return "(empty â€” this is your first experiment)"
    top = sorted(good, key=lambda e: -e["sharpe"])[:top_k]
    lines = []
    for i, e in enumerate(top):
        lines.append(
            f"#{i+1}  iter {e['iter']}  activeSh={e['sharpe']:+.2f}  "
            f"rawSh={e.get('sharpe_raw', 0):+.2f}  beta={e.get('beta', 0):+.2f}  "
            f"consistency={e.get('consistency', 0):.0%}  "
            f"DD={e.get('max_dd', 0):+.1%}"
        )
        if e.get("hypothesis"):
            lines.append(f"    HYP: {e['hypothesis'][:150]}")
        code_snip = "\n".join(
            "    " + ln for ln in e["code"].splitlines()[:code_lines])
        lines.append(code_snip)
        if len(e["code"].splitlines()) > code_lines:
            lines.append("    ...")
        lines.append("")
    return "\n".join(lines)

def failures_for_prompt(log, n_reject=5, n_runtime=3):
    fails = [e for e in log if e.get("error")]
    rejects = [e for e in fails
               if any(k in str(e.get("error", ""))
                      for k in ["REJECT", "DEGENERATE", "LOOKAHEAD"])]
    runtime = [e for e in fails if e not in rejects]
    chosen = rejects[-n_reject:] + runtime[-n_runtime:]
    if not chosen:
        return ""
    lines = ["\n## Recent failures (avoid these mistakes)"]
    for e in chosen:
        lines.append(f"- iter {e.get('iter', '?')}: {str(e['error'])[:180]}")
        if e.get("hypothesis"):
            lines.append(f"    was: {e['hypothesis'][:120]}")
    return "\n".join(lines)

STAGNATION_EXPLORATION_FAMILIES = [
    "momentum",
    "mean_reversion",
    "volume",
    "volatility",
    "multi_factor",
    "regime",
]

def infer_family(entry):
    family = str(entry.get("family", "") or "").strip().lower()
    if family in {
        "momentum", "mean_reversion", "volume", "volatility",
        "ewm", "regime", "multi_factor",
    }:
        return family
    text = "\n".join([
        str(entry.get("mutation_type", "") or ""),
        str(entry.get("hypothesis", "") or ""),
        str(entry.get("code", "") or ""),
    ]).lower()
    if "ewm(" in text:
        return "ewm"
    if "volume" in text:
        return "volume"
    if "rolling" in text and ".std(" in text:
        return "volatility"
    if "blend" in text or "mix" in text:
        return "multi_factor"
    if "reversion" in text or "-close.pct_change(5)" in text:
        return "mean_reversion"
    if "regime" in text or "stress" in text:
        return "regime"
    return "unknown"

def research_score(entry):
    sharpe = float(entry.get("sharpe", 0.0) or 0.0)
    consistency = float(entry.get("consistency", 0.0) or 0.0)
    min_annual = float(entry.get("min_annual", 0.0) or 0.0)
    beta_penalty = max(0.0, abs(float(entry.get("beta", 0.0) or 0.0)) - 0.10)
    turnover_penalty = max(0.0, float(entry.get("turnover", 0.0) or 0.0) - 0.50)
    dd_penalty = max(0.0, abs(float(entry.get("max_dd", 0.0) or 0.0)) - 0.25)
    return (
        sharpe
        + 0.35 * consistency
        + 0.20 * min_annual
        - 1.00 * beta_penalty
        - 0.50 * turnover_penalty
        - 0.30 * dd_penalty
    )

def is_viable_candidate(entry):
    if entry.get("sharpe") is None:
        return False
    return (
        float(entry.get("sharpe", 0.0) or 0.0) >= MIN_VIABLE_TRAIN_SHARPE
        and research_score(entry) >= MIN_VIABLE_RESEARCH_SCORE
    )

def _successful_log_rows(log):
    return [e for e in log if is_viable_candidate(e)]

def _scored_log_rows(log):
    return [e for e in log if e.get("sharpe") is not None]

def _batch_ids(log_rows):
    return sorted({int(e.get("batch", -1)) for e in log_rows if int(e.get("batch", -1)) >= 0})

def _prompt_parent_rows(log):
    return _successful_log_rows(log)

SEED_TEMPLATE_LIBRARY = [
    {
        "seed_id": "seed_momentum_63",
        "family": "momentum",
        "label": "63d momentum rank",
        "code": """def signal(close, volume):
    feature = close.pct_change(63)
    r = feature.rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out""",
    },
    {
        "seed_id": "seed_reversion_5",
        "family": "mean_reversion",
        "label": "negative 5d reversal",
        "code": """def signal(close, volume):
    feature = -close.pct_change(5)
    r = feature.rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out""",
    },
    {
        "seed_id": "seed_ewm_10_50",
        "family": "ewm",
        "label": "10/50 ewm spread",
        "code": """def signal(close, volume):
    fast = close.ewm(span=10, adjust=False).mean()
    slow = close.ewm(span=50, adjust=False).mean()
    feature = (fast - slow) / slow.replace(0, np.nan)
    r = feature.rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out""",
    },
    {
        "seed_id": "seed_regime_blend",
        "family": "regime",
        "label": "63d momentum + 5d reversal blend",
        "code": """def signal(close, volume):
    mom = close.pct_change(63).rank(axis=1, pct=True) - 0.5
    rev = (-close.pct_change(5)).rank(axis=1, pct=True) - 0.5
    feature = 0.6 * mom + 0.4 * rev
    r = feature.rank(axis=1, pct=True)
    out = (r - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out""",
    },
]
_SEED_SUMMARY_CACHE = None

def seed_summary_for_prompt():
    global _SEED_SUMMARY_CACHE
    if _SEED_SUMMARY_CACHE is not None:
        return _SEED_SUMMARY_CACHE
    lines = []
    for seed in SEED_TEMPLATE_LIBRARY:
        sig, err = run_signal_code(seed["code"], close_train, volume_train, vix_s=vix_train, tnx_s=tnx_train, timeout=20)
        if err:
            lines.append(f"- {seed['seed_id']} | family={seed['family']} | ERROR: {err}")
            continue
        m = backtest(sig, close_train)
        lines.append(
            f"- {seed['seed_id']} | family={seed['family']} | label={seed['label']} | "
            f"score={research_score({'sharpe': m['sharpe_active'], 'consistency': m['consistency'], 'min_annual': m['min_annual'], 'beta': m['beta'], 'turnover': m['avg_turnover'], 'max_dd': m['max_dd_active']}):+.2f} | "
            f"primarySh={m['sharpe_active']:+.2f} | rawSh={m['sharpe_raw']:+.2f} | "
            f"benchSpread={m.get('benchmark_spread_sharpe', 0.0):+.2f} | beta={m['beta']:+.2f}"
        )
    _SEED_SUMMARY_CACHE = "\n".join(lines)
    return _SEED_SUMMARY_CACHE

def _top_families_in_log(log_rows, n=3):
    counts = {}
    for row in _prompt_parent_rows(log_rows):
        fam = infer_family(row)
        if fam == "unknown":
            continue
        counts[fam] = counts.get(fam, 0) + 1
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return [fam for fam, _ in ordered[:n]]

def build_diversity_instructions(log_rows, stagnation_gens=0):
    top_families = _top_families_in_log(log_rows)
    unexplored = [f for f in STAGNATION_EXPLORATION_FAMILIES if f not in top_families]
    if stagnation_gens <= 0:
        return (
            "Propose 3 candidates. At least ONE must come from a family that is not "
            f"currently dominant in the log (dominant: {', '.join(top_families) or 'none yet'}). "
            "The other two may refine the current best area."
        )
    if stagnation_gens <= 3:
        fam_hint = ", ".join(unexplored[:2]) if unexplored else "multi_factor, regime"
        return (
            f"STAGNATION DETECTED: {stagnation_gens} batches without a new best score. "
            "At least TWO candidates must come from families outside the dominant cluster. "
            f"Suggested under-explored families: {fam_hint}. "
            "Do not spend the whole batch on small EWM span tweaks."
        )
    fam_hint = ", ".join(unexplored) if unexplored else "volatility, multi_factor, regime"
    return (
        f"HARD STAGNATION: {stagnation_gens} batches without improvement. "
        "ALL THREE candidates must open new territory. "
        f"Choose from: {fam_hint}. "
        "Do not anchor on the current winner. Structural change is required."
    )

def _prompt_progress_state(log_rows):
    good = sorted(
        _scored_log_rows(log_rows),
        key=lambda e: (int(e.get("batch", -1)), int(e.get("iter", -1))),
    )
    if not good:
        return {"stalled_gens": 0, "best_score": 0.0, "best_gain": 0.0, "generations_run": 0}
    batch_best = {}
    for entry in good:
        batch_id = int(entry.get("batch", -1))
        batch_best[batch_id] = max(batch_best.get(batch_id, -1e9), research_score(entry))
    best_so_far = -1e9
    stalled = 0
    last_gain = 0.0
    for batch_id in sorted(batch_best):
        score = batch_best[batch_id]
        if score > best_so_far + 1e-9:
            last_gain = 0.0 if best_so_far <= -1e8 else score - best_so_far
            best_so_far = score
            stalled = 0
        else:
            stalled += 1
            last_gain = score - best_so_far
    return {
        "stalled_gens": stalled,
        "best_score": 0.0 if best_so_far <= -1e8 else float(best_so_far),
        "best_gain": float(last_gain),
        "generations_run": len(batch_best),
    }

def build_stagnation_status(stagnation_gens, best_score, best_score_delta, generations_run):
    if stagnation_gens <= 0:
        return f"No stagnation. Best research score: {best_score:.3f} (improved in the latest scored batch)."
    return (
        f"Stagnation: {stagnation_gens} batches without improvement in best research score.\n"
        f"Best research score so far: {best_score:.3f} (last gain: {best_score_delta:+.4f}).\n"
        f"Total scored batches: {generations_run}.\n"
        "The search is spending too much time in familiar territory. A family-level change is required."
    )

def current_best_for_prompt(log_rows):
    good = _prompt_parent_rows(log_rows)
    if not good:
        return "(no viable parents yet - mutate one of the deterministic seed parents instead of mutating losers)"
    best = max(good, key=research_score)
    return (
        f"iter {best.get('iter', '?')} | family={infer_family(best)} | "
        f"score={research_score(best):+.2f} | primarySh={float(best.get('sharpe', 0.0) or 0.0):+.2f} | "
        f"consistency={float(best.get('consistency', 0.0) or 0.0):.0%} | "
        f"min_annual={float(best.get('min_annual', 0.0) or 0.0):+.2f} | "
        f"beta={float(best.get('beta', 0.0) or 0.0):+.2f}"
    )

def family_summary_for_prompt(log_rows):
    good = _prompt_parent_rows(log_rows)
    if not good:
        return "(no viable families yet)"
    counts = {}
    for entry in good:
        fam = infer_family(entry)
        counts[fam] = counts.get(fam, 0) + 1
    ordered = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return "\n".join([f"- {fam}: {count}" for fam, count in ordered[:6]])

def failure_bucket_counts(log_rows):
    counts = {}
    for entry in log_rows[-40:]:
        err = str(entry.get("error", "") or "").lower()
        if err:
            if "lookahead" in err:
                key = "lookahead"
            elif "structured_parse_failed" in err:
                key = "format_failure"
            elif "duplicate" in err:
                key = "duplicate_candidate"
            elif "semantic_family" in err:
                key = "semantic_family"
            elif "non_viable" in err:
                key = "non_viable_alpha"
            elif "degenerate" in err or "near-constant" in err:
                key = "degenerate_signal"
            elif "validation" in err:
                key = "validation"
            elif "timeout" in err:
                key = "timeout"
            elif "backtest" in err:
                key = "backtest_error"
            else:
                key = "runtime_error"
        elif entry.get("sharpe") is not None:
            if abs(float(entry.get("beta", 0.0) or 0.0)) > 0.10:
                key = "beta_drift"
            elif float(entry.get("turnover", 0.0) or 0.0) > 0.50:
                key = "high_turnover"
            elif float(entry.get("min_annual", 0.0) or 0.0) < 0.0:
                key = "year_instability"
            elif float(entry.get("consistency", 0.0) or 0.0) < 0.50:
                key = "low_consistency"
            else:
                continue
        else:
            continue
        counts[key] = counts.get(key, 0) + 1
    return counts

def cluster_summary_for_prompt(log_rows):
    good = sorted(_prompt_parent_rows(log_rows), key=research_score, reverse=True)
    if not good:
        return "(no viable clusters yet)"
    by_family = {}
    for entry in good:
        fam = infer_family(entry)
        by_family.setdefault(fam, []).append(entry)
    lines = []
    for fam, items in sorted(by_family.items(), key=lambda item: (-len(item[1]), item[0]))[:6]:
        top = max(items, key=research_score)
        lines.append(
            f"- {fam}: count={len(items)} topScore={research_score(top):+.2f} "
            f"topSh={float(top.get('sharpe', 0.0) or 0.0):+.2f} "
            f"cons={float(top.get('consistency', 0.0) or 0.0):.0%}"
        )
    return "\n".join(lines)

def should_abort_early_negative(log_rows):
    evaluated = sorted(
        _scored_log_rows(log_rows),
        key=lambda e: int(e.get("iter", 10**9)),
    )
    n = max(0, int(EARLY_ABORT_NEGATIVE_SUCCESSFUL))
    if (
        n <= 0
        or len(evaluated) < n
        or len({int(e.get("batch", -1)) for e in evaluated}) < EARLY_ABORT_MIN_BATCH
        or int(RUNTIME_STATE.get("reflection_calls", 0) or 0) < EARLY_ABORT_MIN_REFLECTIONS
    ):
        return False, ""
    first_n = evaluated[:n]
    all_bad = all(
        float(row.get("sharpe", 0.0) or 0.0) <= EARLY_ABORT_NEGATIVE_SHARPE
        and research_score(row) <= EARLY_ABORT_NEGATIVE_SCORE
        for row in first_n
    )
    if not all_bad:
        return False, ""
    sample = ", ".join(
        f"iter {row.get('iter', '?')}: Sh={float(row.get('sharpe', 0.0) or 0.0):+.2f}, score={research_score(row):+.2f}"
        for row in first_n[:4]
    )
    reason = (
        f"first {n} scored backtests all <= thresholds "
        f"(Sh <= {EARLY_ABORT_NEGATIVE_SHARPE:+.2f}, score <= {EARLY_ABORT_NEGATIVE_SCORE:+.2f}); "
        f"sample: {sample}"
    )
    return True, reason

def should_abort_search_collapse(log_rows):
    if (
        len({int(e.get("batch", -1)) for e in log_rows}) < SEARCH_COLLAPSE_MIN_BATCHES
        or int(RUNTIME_STATE.get("reflection_calls", 0) or 0) < 1
    ):
        return False, ""
    scored = _scored_log_rows(log_rows)
    collapse_errors = []
    for row in log_rows:
        err = str(row.get("error", "") or "").lower()
        if any(tag in err for tag in ["structured_parse_failed", "duplicate", "semantic_family"]):
            collapse_errors.append(row)
    if len(collapse_errors) < SEARCH_COLLAPSE_MIN_ERRORS:
        return False, ""
    total_attempts = len(scored) + len(collapse_errors)
    if total_attempts <= 0:
        return False, ""
    failure_rate = len(collapse_errors) / total_attempts
    if failure_rate < SEARCH_COLLAPSE_FAILURE_RATE:
        return False, ""
    sample = ", ".join(
        str(row.get("error", "") or "").split("\n", 1)[0][:80]
        for row in collapse_errors[:4]
    )
    reason = (
        f"search-quality collapse: {len(collapse_errors)}/{total_attempts} candidate attempts failed "
        f"due to format/duplicate/semantic-family issues (rate={failure_rate:.0%}); sample: {sample}"
    )
    return True, reason

def should_abort_stagnation(log_rows):
    window = max(0, int(STAGNATION_WINDOW_BATCHES))
    if window <= 0:
        return False, ""
    batch_ids = _batch_ids(log_rows)
    if (
        len(batch_ids) < window
        or int(RUNTIME_STATE.get("reflection_calls", 0) or 0) < STAGNATION_MIN_REFLECTIONS
    ):
        return False, ""
    recent_batches = set(batch_ids[-window:])
    recent_rows = [row for row in log_rows if int(row.get("batch", -1)) in recent_batches]
    recent_scored = _scored_log_rows(recent_rows)
    if not recent_scored:
        return False, ""
    prior_scored = [row for row in _scored_log_rows(log_rows) if int(row.get("batch", -1)) not in recent_batches]
    if not prior_scored:
        return False, ""
    prev_best = max(research_score(row) for row in prior_scored)
    recent_best_row = max(recent_scored, key=research_score)
    recent_best = research_score(recent_best_row)
    improvement = recent_best - prev_best
    failure_rows = [row for row in recent_rows if str(row.get("error", "") or "").strip()]
    if len(failure_rows) < STAGNATION_MIN_FAILURES:
        return False, ""
    total_recent = len(recent_rows)
    if total_recent <= 0:
        return False, ""
    failure_rate = len(failure_rows) / total_recent
    if improvement >= STAGNATION_MIN_SCORE_DELTA or failure_rate < STAGNATION_FAILURE_RATE:
        return False, ""
    sample = ", ".join(
        str(row.get("error", "") or "ok").split("\n", 1)[0][:80]
        for row in failure_rows[:4]
    )
    reason = (
        f"stagnation: last {window} batches improved best score by only {improvement:+.2f} "
        f"(< {STAGNATION_MIN_SCORE_DELTA:+.2f}) with {len(failure_rows)}/{total_recent} "
        f"failed/non-viable attempts (rate={failure_rate:.0%}); "
        f"recent best iter {recent_best_row.get('iter', '?')} score={recent_best:+.2f} "
        f"Sh={float(recent_best_row.get('sharpe', 0.0) or 0.0):+.2f}; sample: {sample}"
    )
    return True, reason

def log_summary_for_prompt(log, top_k=5, code_lines=6):
    good = _prompt_parent_rows(log)
    if not good:
        return "(no viable candidates yet - do not mutate losing rows)"
    top = sorted(good, key=research_score, reverse=True)[:top_k]
    lines = []
    for i, e in enumerate(top):
        lines.append(
            f"#{i+1}  iter {e['iter']}  family={infer_family(e)}  score={research_score(e):+.2f}  "
            f"primarySh={float(e.get('sharpe', 0.0) or 0.0):+.2f}  "
            f"beta={float(e.get('beta', 0.0) or 0.0):+.2f}  "
            f"consistency={float(e.get('consistency', 0.0) or 0.0):.0%}  "
            f"minYr={float(e.get('min_annual', 0.0) or 0.0):+.2f}  "
            f"DD={float(e.get('max_dd', 0.0) or 0.0):+.1%}"
        )
        if e.get("hypothesis"):
            lines.append(f"    HYP: {e['hypothesis'][:150]}")
        if e.get("robustness_rationale"):
            lines.append(f"    ROBUST: {e['robustness_rationale'][:150]}")
        code_snip = "\n".join(
            "    " + ln for ln in e["code"].splitlines()[:code_lines])
        lines.append(code_snip)
        if len(e["code"].splitlines()) > code_lines:
            lines.append("    ...")
        lines.append("")
    return "\n".join(lines)

def failures_for_prompt(log, n_reject=5, n_runtime=3):
    fails = [e for e in log if e.get("error")]
    rejects = [e for e in fails
               if any(k in str(e.get("error", ""))
                      for k in ["REJECT", "DEGENERATE", "LOOKAHEAD"])]
    runtime = [e for e in fails if e not in rejects]
    chosen = rejects[-n_reject:] + runtime[-n_runtime:]
    if not chosen:
        return ""
    lines = ["\n## Recent failures (avoid these mistakes)"]
    for e in chosen:
        lines.append(f"- iter {e.get('iter', '?')}: {str(e['error'])[:180]}")
        if e.get("hypothesis"):
            lines.append(f"    was: {e['hypothesis'][:120]}")
        if e.get("family"):
            lines.append(f"    family: {e.get('family')}")
    return "\n".join(lines)


## Cell 10 â€” The AutoResearch loop

Each batch:
1. Build prompt from research log + reflection memo + failures
2. Generate 3 candidates via LLM
3. Sandbox â†’ validate â†’ lookahead check â†’ backtest each
4. Append results to log

Every `REFLECT_EVERY` batches the model reviews its full log and writes
a **research memo** â€” what's working, what's not, what to try next.
This memo is included in all future prompts: the model learns from itself.


In [ ]:
from concurrent.futures import ThreadPoolExecutor

_COUNTER = [0]

def process_batch(batch_id, response, close_df, volume_df, t_gen):
    """Parse candidates, sandbox + validate + backtest each."""
    cands, parsed_text, repaired = repair_and_extract_candidates(response)
    update_runtime_state(
        structured_candidate_count=len(cands),
        repair_batches=int(RUNTIME_STATE.get("repair_batches", 0)) + (1 if repaired else 0),
    )
    if len(cands) < 3:
        append_log({"batch": batch_id, "iter": _COUNTER[0],
                     "error": "structured_parse_failed",
                     "raw": response[:700],
                     "repaired": repaired,
                     "parsed_text": parsed_text[:700],
                     "structured_candidate_count": len(cands)})
        update_runtime_state(format_failures=int(RUNTIME_STATE.get("format_failures", 0) or 0) + 1)
        _COUNTER[0] += 1
        print(f"  [b{batch_id:02d}] structured parse failed ({len(cands)}/3 candidates)")
        return

    
    existing_fingerprints = {
        e.get("code_fingerprint")
        for e in load_log()
        if e.get("code_fingerprint")
    }
    batch_fingerprints = set()

    gate_mode = globals().get("AUTORESEARCH_LLM_GATE_MODE", "staged")

    for cand in cands:
        it = _COUNTER[0]
        _COUNTER[0] += 1
        fingerprint = code_fingerprint(cand["code"])

        # 1. Duplicate check
        if fingerprint in existing_fingerprints or fingerprint in batch_fingerprints:
            update_runtime_state(duplicate_rejections=int(RUNTIME_STATE.get("duplicate_rejections", 0)) + 1)
            append_log({
                "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
                "parent_id": cand.get("parent_id", "seed"),
                "mutation_type": cand.get("mutation_type", "unspecified"),
                "family": cand.get("family", "unknown"),
                "hypothesis": cand["hypothesis"],
                "code": cand["code"],
                "code_fingerprint": fingerprint,
                "error": "DUPLICATE: repeated candidate fingerprint",
            })
            print(f"  [b{batch_id:02d}.c{cand['idx']}] DUPLICATE")
            continue
        batch_fingerprints.add(fingerprint)

        # 2. Semantic / family validation (Stage A)
        ok, why = validate_candidate_semantics(cand)
        if not ok:
            update_runtime_state(semantic_family_rejections=int(RUNTIME_STATE.get("semantic_family_rejections", 0)) + 1)
            append_log({
                "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
                "parent_id": cand.get("parent_id", "seed"),
                "mutation_type": cand.get("mutation_type", "unspecified"),
                "family": cand.get("family", "unknown"),
                "hypothesis": cand["hypothesis"],
                "code": cand["code"],
                "code_fingerprint": fingerprint,
                "error": f"SEMANTIC_REJECT: {why}",
            })
            print(f"  [b{batch_id:02d}.c{cand['idx']}] SEMANTIC_REJECT")
            continue

        # 3. Stage B - Executable panel precheck
        pre = precheck_candidate(
            cand["code"], close_df, volume_df,
            vix_s=vix_train if "vix_train" in dir() else None,
            tnx_s=tnx_train if "tnx_train" in dir() else None
        )

        append_log({
            "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
            "parent_id": cand.get("parent_id", "seed"),
            "mutation_type": cand.get("mutation_type", "unspecified"),
            "family": cand.get("family", "unknown"),
            "hypothesis": cand["hypothesis"],
            "code": cand["code"],
            "code_fingerprint": fingerprint,
            "precheck": pre,
        })

        if not pre.get("is_executable", False):
            update_runtime_state(precheck_failures=int(RUNTIME_STATE.get("precheck_failures", 0)) + 1)
            append_log({
                "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
                "error": f"PRECHECK_FAILED: {pre.get('failure_class')}",
                "failure_class": pre.get("failure_class"),
                "failure_detail": pre.get("failure_detail"),
            })
            print(f"  [b{batch_id:02d}.c{cand['idx']}] PRECHECK_FAILED {pre.get('failure_class')}")
            continue

        # 4. Tiering + Staged Gate (expensive eval control)
        tier = "promising"
        act = pre.get("activity_stats", {}) or {}
        if act.get("panel_std", 0) < 0.035 or act.get("cross_sectional_dispersion", 0) < 0.025:
            tier = "weak"

        cand["_v3_exploration_tier"] = tier

        if gate_mode == "staged":
            try:
                from autoresearch_v3_helpers import should_run_full_backtest
                if not should_run_full_backtest(pre, exploration_tier=tier, gate_mode="staged"):
                    append_log({
                        "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
                        "skipped_full_backtest": True,
                        "exploration_tier": tier,
                        "gate_mode": "staged",
                    })
                    print(f"  [b{batch_id:02d}.c{cand['idx']}] GATED (tier={tier})")
                    continue
            except Exception as e:
                append_log({
                    "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
                    "error": f"GATE_HELPER_ERROR: {str(e)[:200]}"
                })
                continue

        # 5. Expensive path - only reached by candidates that passed the gate
        t0 = time.time()
        try:
            sig_df, err = run_signal_code(
                cand["code"], close_df, volume_df,
                vix_s=vix_train if "vix_train" in dir() else None,
                tnx_s=tnx_train if "tnx_train" in dir() else None
            )
        except Exception as e:
            append_log({"batch": batch_id, "iter": it, "cand_idx": cand["idx"], "error": f"RUNTIME: {str(e)[:300]}"})
            continue

        if err or sig_df is None:
            append_log({"batch": batch_id, "iter": it, "cand_idx": cand["idx"], "error": f"EXEC_ERROR: {err}"})
            continue

        # Lookahead detection
        try:
            la_ok, la_msg = detect_lookahead(
                cand["code"], close_df, volume_df,
                vix_s=vix_train if "vix_train" in dir() else None,
                tnx_s=tnx_train if "tnx_train" in dir() else None
            )
            if not la_ok:
                append_log({
                    "batch": batch_id, "iter": it, "cand_idx": cand["idx"],
                    "error": f"LOOKAHEAD: {la_msg}"
                })
                continue
        except Exception:
            pass  # Non-fatal

        # Full backtest
        try:
            bt = backtest(sig_df, close_df, cost_bps=5.0)
        except Exception as e:
            append_log({"batch": batch_id, "iter": it, "cand_idx": cand["idx"], "error": f"BACKTEST_ERROR: {str(e)[:300]}"})
            continue

        # Viability + logging (existing scoring logic continues from here)
        # We keep the original scoring / robustness / logging code after this point.
        elapsed = time.time() - t0

        # Basic viability record (simplified for integration repair)
        record = {
            "batch": batch_id,
            "iter": it,
            "cand_idx": cand["idx"],
            "parent_id": cand.get("parent_id", "seed"),
            "mutation_type": cand.get("mutation_type", "unspecified"),
            "family": cand.get("family", "unknown"),
            "hypothesis": cand["hypothesis"],
            "code": cand["code"],
            "code_fingerprint": fingerprint,
            "exploration_tier": tier,
            "gate_mode": gate_mode,
            "train_sharpe": bt.get("sharpe_raw", 0.0),
            "train_score": bt.get("sharpe_active", 0.0),
            "beta": bt.get("beta", 0.0),
            "max_dd": bt.get("max_dd", 0.0),
            "avg_turnover": bt.get("avg_turnover", 0.0),
            "consistency": bt.get("consistency", 0.0),
            "elapsed": round(elapsed, 2),
        }

        # Basic viability gate (can be made stricter later)
        if record["train_sharpe"] > -0.1:   # very permissive for staged exploration
            append_log(record)
            print(f"  [b{batch_id:02d}.c{cand['idx']}] OK (sharpe={record['train_sharpe']:.2f}, tier={tier})")
        else:
            append_log({**record, "error": f"NON_VIABLE: train_sharpe {record['train_sharpe']:.2f}"})
            print(f"  [b{batch_id:02d}.c{cand['idx']}] NON_VIABLE (sharpe={record['train_sharpe']:.2f})")

    # End of batch processing
    return


## Cell 11 â€” Held-out evaluation (2023â€“2024)

Take the top-K train signals and re-run on untouched test data.
Walk-forward: slice held-out into quarters and check consistency.


In [ ]:
TOP_K = 5
HELDOUT_SHORTLIST_K = 20
HELDOUT_MAX_PER_CLUSTER = 2
HELDOUT_MIN_DETERMINISTIC = 5
HELDOUT_MIN_EVOLUTION = 10
HELDOUT_MIN_LLM = 10
HELDOUT_MIN_EVOLUTION_CHAMPIONS = 5
HELDOUT_MAX_PER_MUTATION = 4
COST_STRESS_BPS = (5.0, 10.0, 15.0)
ENSEMBLE_TOP_N = 3
ECONOMIC_SHARPE_FLOOR = globals().get("ECONOMIC_SHARPE_FLOOR", 0.50)
ECONOMIC_EDGE_OVER_DETERMINISTIC = globals().get("ECONOMIC_EDGE_OVER_DETERMINISTIC", 0.05)
APPROVED_WINNER_EDGE_OVER_DETERMINISTIC = globals().get("APPROVED_WINNER_EDGE_OVER_DETERMINISTIC", 0.10)
WF_WINDOWS = max(1, int(globals().get("WF_WINDOWS", 4) or 4))
CHRONOLOGICAL_HOLDOUT_SEGMENTS = max(0, int(globals().get("CHRONOLOGICAL_HOLDOUT_SEGMENTS", 3) or 0))
ROLLING_HOLDOUT_SEGMENTS = max(0, int(globals().get("ROLLING_HOLDOUT_SEGMENTS", 3) or 0))
ROLLING_HOLDOUT_MIN_POINTS = max(20, int(globals().get("ROLLING_HOLDOUT_MIN_POINTS", 126) or 126))
SUBPERIOD_WINDOWS = (
    ("2015-2018", "2015-01-01", "2018-12-31"),
    ("2019-2021", "2019-01-01", "2021-12-31"),
    ("2022-2024", "2022-01-01", "2024-12-31"),
)
EVOLUTION_DEEP_DIVE_FILE = OUT / "evolution_deep_dive.md"
EVOLUTION_TLDR_FILE = OUT / "evolution_tldr.md"
# â”€â”€ Remediation V3: Staged LLM Exploration Gate â”€â”€
AUTORESEARCH_LLM_GATE_MODE = globals().get("AUTORESEARCH_LLM_GATE_MODE", "staged")  # "staged" | "strict"
LLM_EXPLORATION_MIN_EXECUTABLE = 20
LLM_EXPLORATION_MIN_FAMILIES = 3
LLM_EXECUTION_STAGNATION_THRESHOLD = 0.15  # 15% executable rate
LLM_EXECUTION_STAGNATION_WINDOW = 6
LLM_ALPHA_STAGNATION_IMPROVEMENT = 0.05
LLM_ALPHA_STAGNATION_MIN_EXEC = 20

# Stage B thresholds (executable panel precheck)
STAGE_B_MIN_ACTIVITY_STD = 0.01
STAGE_B_MIN_CROSS_SECTIONAL_DISPERSION = 0.02
STAGE_B_MAX_DIRECTIONAL_BIAS = 0.85



def _float_or(value, default=0.0):
    try:
        if value is None:
            return default
        return float(value)
    except Exception:
        return default


def _safe_json_value(path, default):
    if path is None:
        return default
    try:
        if (not path.exists()) or path.stat().st_size == 0:
            return default
        text = path.read_text()
        if not text.strip():
            return default
        value = json.loads(text)
        return default if value is None else value
    except Exception:
        return default


def _as_row_list(value):
    if isinstance(value, list):
        return [r for r in value if isinstance(r, dict)]
    if isinstance(value, dict):
        for key in ("rows", "results", "candidates", "lineage", "programs"):
            rows = value.get(key)
            if isinstance(rows, list):
                return [r for r in rows if isinstance(r, dict)]
        return [value] if value.get("score") is not None else []
    return []


def _safe_json_rows(path):
    return _as_row_list(_safe_json_value(path, []))


def _safe_json_dict(path):
    value = _safe_json_value(path, {})
    return value if isinstance(value, dict) else {}


def _runtime_scope_metadata():
    path = globals().get("RUNTIME_METADATA_FILE")
    return _safe_json_dict(path) if path is not None else {}


def _runtime_family_scope_lists():
    runtime_metadata = _runtime_scope_metadata()
    configured = runtime_metadata.get("configured_method_families")
    if not isinstance(configured, list) or not configured:
        configured = list(globals().get("ROADMAP_METHOD_FAMILIES", globals().get("BENCHMARK_METHOD_FAMILIES", [])))
    executed = runtime_metadata.get("executed_method_families")
    if not isinstance(executed, list) or not executed:
        executed = list(globals().get("EXECUTED_METHOD_FAMILIES", []))
    configured = [str(x) for x in configured]
    executed = [str(x) for x in executed]
    deferred = [family for family in configured if family not in executed]
    active_sources = runtime_metadata.get("active_result_sources")
    if not isinstance(active_sources, list) or not active_sources:
        active_sources = list(globals().get("ACTIVE_RESULT_SOURCES", ["deterministic", "parameter_search_evolution"]))
    active_scope = runtime_metadata.get("active_execution_scope") or globals().get(
        "ACTIVE_EXECUTION_SCOPE",
        "deterministic/classical/autoresearch only",
    )
    return {
        "configured": configured,
        "executed": executed,
        "deferred": deferred,
        "active_sources": [str(x) for x in active_sources],
        "active_scope": str(active_scope),
    }


def _row_family_tokens(row):
    if not isinstance(row, dict):
        return set()
    keys = (
        "source",
        "family",
        "base_family",
        "benchmark_family",
        "model_family",
        "study_family",
        "study_type",
        "label",
        "name",
    )
    tokens = set()
    for key in keys:
        value = row.get(key)
        if value in (None, ""):
            continue
        text = str(value).strip().lower()
        if text:
            tokens.add(text)
    return tokens


def _row_matches_deferred_stage_family(row):
    deferred_aliases = {
        "lstm",
        "lstm_sharpe",
        "tabular",
        "tabular_ml",
        "gbt",
        "gradient_boosted_trees",
        "combination",
        "combination_studies",
        "ensemble_combination",
    }
    tokens = _row_family_tokens(row)
    return any(alias in token for token in tokens for alias in deferred_aliases)


def _partial_report_path():
    path = globals().get("PARTIAL_REPORT_FILE")
    if path is None and "OUT" in globals():
        path = OUT / "partial_run_report.json"
    return path


def _load_partial_run_report():
    return _safe_json_dict(_partial_report_path())


def _row_identity_safe(row, default_prefix="row"):
    helper = globals().get("row_identity")
    if callable(helper):
        try:
            return str(helper(row, default_prefix=default_prefix))
        except TypeError:
            try:
                return str(helper(row, default_prefix))
            except Exception:
                pass
        except Exception:
            pass
    if not isinstance(row, dict):
        return f"{default_prefix}:unknown"
    for key in ("parent_id", "iter", "program_hash", "program_sig", "signature", "cluster_id"):
        value = row.get(key)
        if value not in (None, ""):
            return str(value)
    return f"{default_prefix}:unknown"


def _heldout_iter_label(row):
    if not isinstance(row, dict):
        return "row:unknown"
    source = row.get("source", "unknown")
    signature = str(row.get("signature") or row.get("program_hash") or row.get("program_sig") or "nosig")[:8]
    cluster_id = str(row.get("cluster_id", "unknown"))
    if source == "parameter_search_evolution":
        return f"evo:{cluster_id}:{signature}"
    if source == "deterministic":
        return str(row.get("parent_id") or _row_identity_safe(row, default_prefix="det"))
    return _row_identity_safe(row, default_prefix=str(source or default_prefix))


def _selection_score_safe(sharpe, wf_median, consistency, beta, avg_turnover, wf_min=0.0, max_dd=0.0):
    try:
        return selection_score(
            sharpe,
            wf_median,
            consistency,
            beta,
            avg_turnover,
            wf_min=wf_min,
            max_dd=max_dd,
        )
    except TypeError:
        return selection_score(sharpe, wf_median, consistency, beta, avg_turnover)


def _robustness_score_safe(metric_row):
    helper = globals().get("robustness_score")
    if callable(helper):
        try:
            return helper(metric_row)
        except Exception:
            return None
    return None


def _median_value(values):
    vals = sorted([float(v) for v in values if v is not None])
    if not vals:
        return None
    mid = len(vals) // 2
    if len(vals) % 2:
        return vals[mid]
    return 0.5 * (vals[mid - 1] + vals[mid])


def _window_median_sharpe(windows):
    if not isinstance(windows, list):
        return None
    return _median_value([_float_or(w.get("sharpe"), None) for w in windows if isinstance(w, dict)])


def _cost_stress_avg_sharpe(row):
    stress = row.get("cost_stress", {}) if isinstance(row, dict) else {}
    vals = []
    if isinstance(stress, dict):
        for payload in stress.values():
            if isinstance(payload, dict):
                vals.append(_float_or(payload.get("sharpe"), None))
    vals = [v for v in vals if v is not None]
    if not vals:
        return None
    return float(sum(vals) / len(vals))


def _build_chronological_window_specs(index_like, segments, min_points=60):
    index = list(index_like) if index_like is not None else []
    n = len(index)
    if segments <= 0 or n < min_points:
        return []
    window_len = max(n // segments, min_points)
    if window_len > n:
        return []
    specs = []
    lo = 0
    slot = 1
    while lo < n and len(specs) < segments:
        hi = n if len(specs) == segments - 1 else min(n, lo + window_len)
        if hi - lo < min_points:
            break
        specs.append(
            {
                "label": f"chrono_{slot}",
                "lo": lo,
                "hi": hi,
                "start": str(index[lo])[:10],
                "end": str(index[hi - 1])[:10],
            }
        )
        lo = hi
        slot += 1
    return specs


def _build_rolling_window_specs(index_like, segments, min_points=126):
    index = list(index_like) if index_like is not None else []
    n = len(index)
    if segments <= 0 or n < min_points:
        return []
    window_len = max(min_points, n // max(segments, 1))
    if window_len > n:
        return []
    max_start = max(n - window_len, 0)
    if segments == 1:
        starts = [max_start]
    else:
        stride = max(max_start // max(segments - 1, 1), 1)
        starts = []
        for i in range(segments):
            starts.append(min(i * stride, max_start))
        starts = sorted(set(starts))
        if starts[-1] != max_start:
            starts.append(max_start)
    specs = []
    for idx, lo in enumerate(starts, start=1):
        hi = min(n, lo + window_len)
        if hi - lo < min_points:
            continue
        specs.append(
            {
                "label": f"roll_{idx}",
                "lo": lo,
                "hi": hi,
                "start": str(index[lo])[:10],
                "end": str(index[hi - 1])[:10],
            }
        )
        if len(specs) >= segments:
            break
    return specs


def _supporting_diagnostic_report(candidate, baseline):
    diagnostics = []
    if not isinstance(candidate, dict) or not isinstance(baseline, dict):
        return {"diagnostics": diagnostics, "wins": 0, "losses": 0, "ties": 0, "comparable": 0, "not_losing_majority": False}

    metric_specs = (
        ("test_score", "composite score", _float_or(candidate.get("test_score"), None), _float_or(baseline.get("test_score"), None)),
        ("test_sharpe", "test Sharpe", _float_or(candidate.get("test_sharpe"), None), _float_or(baseline.get("test_sharpe"), None)),
        ("wf_median", "WF median", _float_or(candidate.get("wf_median"), None), _float_or(baseline.get("wf_median"), None)),
        ("wf_min", "WF min", _float_or(candidate.get("wf_min"), None), _float_or(baseline.get("wf_min"), None)),
        (
            "test_robustness_score",
            "robustness score",
            _float_or(candidate.get("test_robustness_score", candidate.get("train_robustness_score")), None),
            _float_or(baseline.get("test_robustness_score", baseline.get("train_robustness_score")), None),
        ),
        (
            "test_benchmark_spread_sharpe",
            "benchmark-spread Sharpe",
            _float_or(candidate.get("test_benchmark_spread_sharpe"), None),
            _float_or(baseline.get("test_benchmark_spread_sharpe"), None),
        ),
        ("cost_stress_avg_sharpe", "cost-stress avg Sharpe", _cost_stress_avg_sharpe(candidate), _cost_stress_avg_sharpe(baseline)),
        ("subperiod_median_sharpe", "subperiod median Sharpe", _window_median_sharpe(candidate.get("subperiods")), _window_median_sharpe(baseline.get("subperiods"))),
        (
            "chronological_median_sharpe",
            "chronological holdout median Sharpe",
            _window_median_sharpe(candidate.get("chronological_holdout")),
            _window_median_sharpe(baseline.get("chronological_holdout")),
        ),
        (
            "rolling_median_sharpe",
            "rolling holdout median Sharpe",
            _window_median_sharpe(candidate.get("rolling_holdout")),
            _window_median_sharpe(baseline.get("rolling_holdout")),
        ),
    )

    wins = 0
    losses = 0
    ties = 0
    for key, label, candidate_value, baseline_value in metric_specs:
        if candidate_value is None or baseline_value is None:
            continue
        delta = float(candidate_value - baseline_value)
        if abs(delta) <= 1e-9:
            verdict = "tie"
            ties += 1
        elif delta > 0:
            verdict = "win"
            wins += 1
        else:
            verdict = "loss"
            losses += 1
        diagnostics.append(
            {
                "key": key,
                "label": label,
                "candidate": float(candidate_value),
                "baseline": float(baseline_value),
                "delta": delta,
                "verdict": verdict,
            }
        )

    comparable = wins + losses + ties
    return {
        "diagnostics": diagnostics,
        "wins": wins,
        "losses": losses,
        "ties": ties,
        "comparable": comparable,
        "not_losing_majority": bool(comparable and losses * 2 <= comparable),
    }


def _heldout_rule_reason_lines(heldout_report):
    if not isinstance(heldout_report, dict):
        return ["held-out winner rule not evaluated"]
    reasons = []
    leader = heldout_report.get("leader")
    baseline = heldout_report.get("best_deterministic")
    comparison = heldout_report.get("comparison", {})
    if leader is None:
        return ["held-out evaluation produced no leader"]
    if baseline is None:
        reasons.append("no deterministic baseline available for held-out comparison")
    elif leader.get("source") == "deterministic":
        reasons.append("highest composite-score candidate is still the deterministic baseline")
    edge = heldout_report.get("score_edge_vs_deterministic")
    required_edge = heldout_report.get("required_edge", APPROVED_WINNER_EDGE_OVER_DETERMINISTIC)
    if edge is None:
        reasons.append("score edge vs deterministic baseline unavailable")
    elif edge < required_edge:
        reasons.append(f"composite-score edge {edge:+.2f} is below required {required_edge:+.2f}")
    if not comparison.get("not_losing_majority"):
        reasons.append(
            "supporting diagnostics lost majority "
            f"({comparison.get('losses', 0)} losses across {comparison.get('comparable', 0)} comparable diagnostics)"
        )
    return reasons or ["winner rule met"]


def _build_heldout_report(test_rows):
    rows = [r for r in test_rows if isinstance(r, dict)]
    ranked = sorted(rows, key=lambda r: -_float_or(r.get("test_score"), -999.0))
    leader = ranked[0] if ranked else None
    best_deterministic = max(
        [r for r in rows if r.get("source") == "deterministic"],
        key=lambda r: _float_or(r.get("test_score"), -999.0),
        default=None,
    )
    best_evolution = max(
        [r for r in rows if r.get("source") == "parameter_search_evolution"],
        key=lambda r: _float_or(r.get("test_score"), -999.0),
        default=None,
    )
    score_edge = None
    if leader and best_deterministic:
        score_edge = float(_float_or(leader.get("test_score")) - _float_or(best_deterministic.get("test_score")))
    comparison = _supporting_diagnostic_report(leader, best_deterministic)
    edge_ok = bool(best_deterministic and score_edge is not None and score_edge >= APPROVED_WINNER_EDGE_OVER_DETERMINISTIC)
    source_ok = bool(leader and leader.get("source") != "deterministic")
    diagnostics_ok = bool(comparison.get("not_losing_majority"))
    winner = leader if leader and source_ok and edge_ok and diagnostics_ok else None
    return {
        "ranked": ranked,
        "leader": leader,
        "winner": winner,
        "best_deterministic": best_deterministic,
        "best_evolution": best_evolution,
        "score_edge_vs_deterministic": score_edge,
        "required_edge": APPROVED_WINNER_EDGE_OVER_DETERMINISTIC,
        "comparison": comparison,
        "reasons": _heldout_rule_reason_lines(
            {
                "leader": leader,
                "best_deterministic": best_deterministic,
                "comparison": comparison,
                "score_edge_vs_deterministic": score_edge,
                "required_edge": APPROVED_WINNER_EDGE_OVER_DETERMINISTIC,
            }
        ),
        "status": "approved_winner" if winner else "no_winner_yet",
    }


def walk_forward(code_str, close_df, volume_df, flipped=False, n_windows=WF_WINDOWS):
    N = len(close_df)
    sz = max(N // n_windows, 1)
    per_window = []
    for w in range(n_windows):
        lo = w * sz
        hi = (w + 1) * sz if w < n_windows - 1 else N
        sub_c, sub_v = close_df.iloc[lo:hi], volume_df.iloc[lo:hi]
        sig, err = run_signal_code(code_str, sub_c, sub_v, vix_s=vix_test.iloc[lo:hi] if vix_test is not None else None, tnx_s=tnx_test.iloc[lo:hi] if tnx_test is not None else None, timeout=20)
        if err:
            per_window.append({"window": w, "error": err})
            continue
        if flipped:
            sig = -sig
        m = backtest(sig, sub_c)
        per_window.append({"window": w, "sharpe": m.get("sharpe", 0.0), "benchmark_spread_sharpe": m.get("benchmark_spread_sharpe", m.get("excess_sharpe", 0.0)), "ann_return": m.get("ann_return", 0.0), "beta": m.get("beta", 0.0), "max_dd": m.get("max_dd", 0.0), "turnover": m.get("avg_turnover", 0.0)})
    return per_window


def deterministic_code(row):
    if row.get("code"):
        return row["code"]
    short_span = row["short_span"]
    long_span = row["long_span"]
    variant = row["mutation_type"]
    params = normalize_aux_params(row.get("params", {}))
    vol_window = int(params.get("vol_window", 20))
    vol_gate_window = int(params.get("vol_gate_window", 20))
    vol_cap = float(params.get("vol_cap", 3.0))
    vix_threshold = float(params.get("vix_threshold", 24.0))
    if variant == "plain":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    out = (core.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "rank_norm":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    out = (core.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "vol_scale":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    vol = close.pct_change().rolling({vol_window}).std().replace(0, np.nan)
    scaled = core.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-3.0, 3.0)
    out = (scaled.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "volume_gate":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    vratio = volume / volume.rolling({vol_gate_window}).mean()
    gated = core * vratio.clip(lower=0.5, upper=1.5)
    out = (gated.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "regime_gate":
        return f"""def signal(close, volume, vix=None, tnx=None):
    fast = close.ewm(span={short_span}, min_periods={short_span}).mean().shift(1)
    slow = close.ewm(span={long_span}, min_periods={long_span}).mean().shift(1)
    core = fast / slow - 1.0
    regime = 1.0 if vix is None else (vix.rolling(5).mean() < {vix_threshold}).astype(float).values[:, None]
    gated = core * regime
    out = (gated.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "ts_momentum":
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span})
    out = (trend.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "short_reversal":
        return f"""def signal(close, volume, vix=None, tnx=None):
    reversal = -close.pct_change({short_span})
    out = (reversal.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant in ("vol_adjusted", "vol_adjusted_momentum"):
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span})
    vol = close.pct_change().rolling({vol_window}).std().replace(0, np.nan)
    scaled = trend.div(vol.clip(lower=1e-6), fill_value=0.0).clip(-{vol_cap}, {vol_cap})
    out = (scaled.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant in ("volume_confirm", "volume_confirmed_momentum"):
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span})
    vratio = volume / volume.rolling({vol_gate_window}).mean()
    confirmed = trend * vratio.clip(lower=0.5, upper=1.8)
    out = (confirmed.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "regime_momentum":
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend_fast = close.pct_change({short_span})
    trend_slow = close.pct_change({long_span})
    trend = trend_fast - trend_slow
    if vix is not None:
        regime = (vix.rolling(5).mean() < {vix_threshold}).astype(float).values[:, None]
        trend = trend * regime
    out = (trend.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    if variant == "multi_factor":
        return f"""def signal(close, volume, vix=None, tnx=None):
    trend = close.pct_change({long_span}).rank(axis=1, pct=True) - 0.5
    reversal = (-close.pct_change({short_span})).rank(axis=1, pct=True) - 0.5
    vol = close.pct_change().rolling({vol_window}).std().replace(0, np.nan)
    low_vol = (-vol).rank(axis=1, pct=True) - 0.5
    combined = 0.55 * trend + 0.25 * reversal + 0.20 * low_vol
    out = (combined.rank(axis=1, pct=True) - 0.5) * 2
    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)
    return out"""

    raise ValueError(variant)


def _oriented_metrics(backtest_row, flipped=False):
    if flipped:
        return {
            "sharpe": backtest_row["sharpe_inverted"],
            "ann_return": backtest_row["ann_return_inverted"],
            "max_dd": backtest_row["max_dd_inverted"],
            "beta": backtest_row["beta_inverted"],
            "consistency": backtest_row["consistency_inverted"],
            "equity": backtest_row["equity_inverted"],
        }
    return {
        "sharpe": backtest_row["sharpe"],
        "ann_return": backtest_row["ann_return"],
        "max_dd": backtest_row["max_dd"],
        "beta": backtest_row["beta"],
        "consistency": backtest_row["consistency"],
        "equity": backtest_row["equity"],
    }


def _concat_optional(train_series, test_series):
    if train_series is None and test_series is None:
        return None
    if train_series is None:
        return test_series
    if test_series is None:
        return train_series
    return pd.concat([train_series, test_series]).sort_index()


def _full_test_panels():
    close_full = globals().get("close_all")
    if close_full is None:
        close_full = pd.concat([close_train, close_test]).sort_index()
    volume_full = globals().get("volume_all")
    if volume_full is None:
        volume_full = pd.concat([volume_train, volume_test]).sort_index()
    vix_full = globals().get("vix_all")
    if vix_full is None:
        vix_full = _concat_optional(globals().get("vix_train"), globals().get("vix_test"))
    tnx_full = globals().get("tnx_all")
    if tnx_full is None:
        tnx_full = _concat_optional(globals().get("tnx_train"), globals().get("tnx_test"))
    return close_full, volume_full, vix_full, tnx_full


def _oriented_signal_code(code_str, flipped=False):
    """Return executable signal code in the same orientation used during evaluation."""
    code_str = str(code_str or "")
    if not flipped:
        return code_str
    if "def signal(" not in code_str:
        return code_str
    raw_code = code_str.replace("def signal(", "def _raw_signal(", 1)
    return (
        raw_code.rstrip()
        + "\n\n\n"
        + "def signal(close, volume, vix=None, tnx=None):\n"
        + "    out = -_raw_signal(close, volume, vix=vix, tnx=tnx)\n"
        + "    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)\n"
        + "    return out\n"
    )


def _ensemble_code(rows):
    blocks = []
    calls = []
    for i, row in enumerate(rows):
        fn_code = row.get("code") or deterministic_code(row)
        if not row.get("code_is_oriented"):
            fn_code = _oriented_signal_code(fn_code, bool(row.get("flipped")))
        fn_code = fn_code.replace("def signal(", f"def _signal_{i}(", 1)
        blocks.append(fn_code)
        calls.append(f"    s{i} = _signal_{i}(close, volume, vix=vix, tnx=tnx)")
    avg_expr = " + ".join([f"s{i}" for i in range(len(rows))])
    meta_lines = "\n".join(
        [
            f"# member_{i + 1}: iter={row.get('iter')} cluster={row.get('cluster_id')} test_sc={row.get('test_score', 0.0):+.2f}"
            for i, row in enumerate(rows)
        ]
    )
    return (
        f"# objective=market_neutral_net_sharpe ensemble_top_n={len(rows)}\n"
        f"{meta_lines}\n\n"
        + "import numpy as np\n\n"
        + "\n\n".join(blocks)
        + "\n\n"
        + "def signal(close, volume, vix=None, tnx=None):\n"
        + "\n".join(calls)
        + "\n"
        + f"    out = ({avg_expr}) / {float(len(rows)):.1f}\n"
        + "    out = out.sub(out.mean(axis=1), axis=0).clip(-1, 1).fillna(0.0)\n"
        + "    return out\n"
    )



def _safe_load_llm_rows_local():
    path = globals().get("LOG_FILE", globals().get("RESEARCH_LOG"))
    if path is None:
        return []
    try:
        if (not path.exists()) or path.stat().st_size == 0:
            return []
    except Exception:
        return []
    rows = []
    try:
        for raw_line in path.read_text().splitlines():
            line = raw_line.strip()
            if not line:
                continue
            try:
                row = json.loads(line)
            except Exception:
                continue
            if not isinstance(row, dict) or not row.get("code"):
                continue
            if row.get("error"):
                continue
            if row.get("score") is None and row.get("research_score") is None:
                continue
            candidate = dict(row)
            candidate["source"] = "llm_autoresearch"
            if candidate.get("score") is None:
                candidate["score"] = candidate.get("research_score")
            if candidate.get("train_score") is None:
                candidate["train_score"] = candidate.get("score")
            if candidate.get("train_sharpe") is None:
                candidate["train_sharpe"] = candidate.get("sharpe")
            if candidate.get("signature") is None:
                candidate["signature"] = candidate.get("code_fingerprint") or candidate.get("parent_id") or candidate.get("iter")
            if candidate.get("cluster_id") is None:
                candidate["cluster_id"] = f"llm:{candidate.get('family', candidate.get('mutation_type', 'unknown'))}"
            if candidate.get("mutation_type") is None:
                candidate["mutation_type"] = candidate.get("family", "llm_signal")
            rows.append(candidate)
    except Exception:
        return []
    return rows
def _safe_load_search_results_local():
    llm_rows = _safe_load_llm_rows_local()
    if "load_search_results" in globals():
        try:
            rows = _as_row_list(load_search_results())
            if rows:
                rows.extend(llm_rows)
                return rows
        except Exception:
            pass
    rows = []
    if "load_parameter_search" in globals():
        try:
            rows.extend(_as_row_list(load_parameter_search()))
        except Exception:
            pass
    elif "PARAM_SEARCH_FILE" in globals() and PARAM_SEARCH_FILE.exists():
        rows.extend(_safe_json_rows(PARAM_SEARCH_FILE))
    if "load_deterministic" in globals():
        try:
            rows.extend(_as_row_list(load_deterministic()))
        except Exception:
            pass
    elif "DETERMINISTIC_FILE" in globals() and DETERMINISTIC_FILE.exists():
        rows.extend(_safe_json_rows(DETERMINISTIC_FILE))
    rows.extend(llm_rows)
    return rows


def _rank_candidates(rows):
    candidates = [r for r in rows if r.get("score") is not None]
    robust = [r for r in candidates if r.get("robust_ok")]
    return sorted(robust if robust else candidates, key=lambda r: (not r.get("robust_ok", False), -_float_or(r.get("score"), -999.0)))


def _pick_diverse(rows, limit, seen_sig=None, per_cluster=None, per_mutation=None):
    picked = []
    seen_sig = seen_sig if seen_sig is not None else set()
    per_cluster = per_cluster if per_cluster is not None else {}
    per_mutation = per_mutation if per_mutation is not None else {}
    for row in _rank_candidates(rows):
        sig = row.get("signature")
        cid = row.get("cluster_id", "unknown")
        mutation_type = row.get("mutation_type", "unknown")
        if sig and sig in seen_sig:
            continue
        if per_cluster.get(cid, 0) >= HELDOUT_MAX_PER_CLUSTER:
            continue
        if per_mutation.get(mutation_type, 0) >= HELDOUT_MAX_PER_MUTATION:
            continue
        picked.append(row)
        if sig:
            seen_sig.add(sig)
        per_cluster[cid] = per_cluster.get(cid, 0) + 1
        per_mutation[mutation_type] = per_mutation.get(mutation_type, 0) + 1
        if len(picked) >= limit:
            break
    return picked


def _evolution_health_lines(evo_summary, partial_report=None):
    partial_report = partial_report if isinstance(partial_report, dict) else {}
    generations = evo_summary.get("generations", []) if isinstance(evo_summary, dict) else []
    generations = [g for g in generations if isinstance(g, dict)]
    eval_total = sum(int(_float_or(g.get("eval_count"), 0.0)) for g in generations)
    valid_total = sum(int(_float_or(g.get("valid_count"), 0.0)) for g in generations)
    robust_total = sum(int(_float_or(g.get("robust_count"), 0.0)) for g in generations)

    valid_rate = partial_report.get("valid_rate", partial_report.get("overall_valid_rate"))
    if valid_rate is None and eval_total:
        valid_rate = valid_total / max(eval_total, 1)

    zero_robust = partial_report.get("zero_robust_generations")
    if zero_robust is None and generations:
        zero_robust = [g.get("gen_id") for g in generations if int(_float_or(g.get("robust_count"), 0.0)) == 0]

    zero_robust_streak = partial_report.get("zero_robust_streak")
    if zero_robust_streak is None and generations:
        zero_robust_streak = 0
        for g in reversed(generations):
            if int(_float_or(g.get("robust_count"), 0.0)) == 0:
                zero_robust_streak += 1
            else:
                break

    lines = []
    if valid_rate is not None:
        try:
            rate_text = f"{float(valid_rate):.1%}"
        except Exception:
            rate_text = str(valid_rate)
        suffix = f" ({valid_total}/{eval_total} valid/evaluated)" if eval_total else ""
        lines.append(f"valid_rate={rate_text}{suffix}")
    if zero_robust is not None:
        if isinstance(zero_robust, list):
            tail = zero_robust[-5:] if zero_robust else []
            lines.append(f"zero_robust_generations={len(zero_robust)} latest={tail}")
        else:
            lines.append(f"zero_robust_generations={zero_robust}")
    if zero_robust_streak:
        lines.append(f"zero_robust_streak={zero_robust_streak}")
    if eval_total:
        lines.append(f"robust_total={robust_total}/{valid_total} valid rows")
    return lines


def _partial_report_line(partial_report):
    if not isinstance(partial_report, dict) or not partial_report:
        return None
    fields = []
    for key in (
        "status",
        "phase",
        "gen_id",
        "generation",
        "generations_executed",
        "valid_rate",
        "robust_count",
        "zero_robust_streak",
        "stop_reason",
        "updated_at",
        "timestamp",
    ):
        if key in partial_report:
            fields.append(f"{key}={partial_report.get(key)}")
    if not fields:
        fields.append("keys=" + ",".join(list(partial_report.keys())[:8]))
    return " ".join(fields)


def _write_evolution_analysis(test_rows):
    evo_summary = _safe_json_dict(globals().get("EVOLUTION_SUMMARY_FILE"))
    partial_report = _load_partial_run_report()
    if not evo_summary and not partial_report:
        return

    search_rows = [r for r in _safe_load_search_results_local() if r.get("score") is not None]
    robust_rows = [r for r in search_rows if r.get("robust_ok")]
    by_cluster = {}
    for r in robust_rows:
        cid = r.get("cluster_id", "unknown")
        prev = by_cluster.get(cid)
        if prev is None or _float_or(r.get("score"), -999.0) > _float_or(prev.get("score"), -999.0):
            by_cluster[cid] = r
    top_clusters = sorted(by_cluster.values(), key=lambda r: -_float_or(r.get("score"), -999.0))[:8]
    generations = [g for g in evo_summary.get("generations", []) if isinstance(g, dict)]
    health_lines = _evolution_health_lines(evo_summary, partial_report)
    partial_line = _partial_report_line(partial_report)

    deep = []
    deep.append("# Evolution Deep Dive\n")
    deep.append("## Run Policy")
    p = evo_summary.get("policy", {}) if isinstance(evo_summary.get("policy"), dict) else {}
    deep.append(
        f"- generations={evo_summary.get('generations_executed')} stop_reason={evo_summary.get('stop_reason')} "
        f"max_generations={p.get('max_generations')} trials_per_generation={p.get('trials_per_generation')} "
        f"beam={p.get('beam_width')} survivors={p.get('survivors')} min_gain={p.get('min_gen_growth')}"
    )
    if health_lines:
        deep.append("\n## Validity / Robustness Watch")
        deep.extend([f"- {line}" for line in health_lines])
    if partial_line:
        deep.append("\n## Partial Run Report")
        deep.append(f"- {partial_line}")
    deep.append("\n## Generation Trace")
    for rr in generations:
        deep.append(
            f"- g{rr.get('gen_id')}: valid={rr.get('valid_count')} robust={rr.get('robust_count')} "
            f"best={rr.get('best_score')} gain={rr.get('improvement_vs_prev_best')} stalled={rr.get('stalled')} "
            f"survivors={rr.get('survivor_count')}"
        )
    deep.append("\n## Best Robust Clusters (Train)")
    for r in top_clusters:
        line = (
            f"- {r.get('cluster_id')}: score={_float_or(r.get('score')):+.2f} "
            f"trainSh={_float_or(r.get('train_sharpe')):+.2f} wf={_float_or(r.get('wf_median')):+.2f}/{_float_or(r.get('wf_min')):+.2f} "
            f"beta={_float_or(r.get('beta')):+.2f} to={_float_or(r.get('turnover')):.2f}"
        )
        if r.get("robustness_score") is not None:
            line += f" robustScore={_float_or(r.get('robustness_score')):.1f}"
        deep.append(line)
    if test_rows:
        heldout_report = _build_heldout_report(test_rows)
        leader = heldout_report.get("leader")
        winner = heldout_report.get("winner")
        best_det = heldout_report.get("best_deterministic")
        comparison = heldout_report.get("comparison", {})
        score_edge = heldout_report.get("score_edge_vs_deterministic")
        best_evo = heldout_report.get("best_evolution")
        evo_edge = None if not (best_det and best_evo) else float(best_evo["test_score"] - best_det["test_score"])
        econ_success = bool(
            best_evo
            and best_evo["test_sharpe"] >= ECONOMIC_SHARPE_FLOOR
            and (best_det is None or evo_edge >= ECONOMIC_EDGE_OVER_DETERMINISTIC)
        )
        deep.append("\n## Held-out Verdict")
        if leader:
            deep.append(
                f"- Leader by composite score: {leader['cluster_id']} ({leader['iter']}) "
                f"score={leader['test_score']:+.2f} testSh={leader['test_sharpe']:+.2f} "
                f"dd={leader['test_dd']:+.1%} beta={leader['test_beta']:+.2f}"
            )
        if best_det:
            deep.append(
                f"- Best deterministic baseline: {best_det['cluster_id']} ({best_det['iter']}) "
                f"score={best_det['test_score']:+.2f} testSh={best_det['test_sharpe']:+.2f}"
            )
        if winner:
            deep.append(
                f"- Approved winner: {winner['cluster_id']} ({winner['iter']}) "
                f"edge_vs_det={_float_or(score_edge):+.2f} diagnostics="
                f"{comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T"
            )
        else:
            deep.append("- Approved winner: no winner yet")
            deep.extend([f"- Rule gap: {reason}" for reason in heldout_report.get("reasons", [])])
        deep.append(
            f"- Winner rule: highest composite score, edge >= {APPROVED_WINNER_EDGE_OVER_DETERMINISTIC:+.2f} "
            "vs best deterministic baseline, and no majority loss across supporting diagnostics."
        )
        deep.append(
            f"- Economic success: {econ_success} "
            f"(requires evolved Sharpe >= {ECONOMIC_SHARPE_FLOOR:.2f} and edge >= {ECONOMIC_EDGE_OVER_DETERMINISTIC:+.2f})"
        )
        if best_det and leader:
            deep.append(
                f"- Leader vs deterministic baseline: leader={leader['test_score']:+.2f}/{leader['test_sharpe']:+.2f} "
                f"det={best_det['test_score']:+.2f}/{best_det['test_sharpe']:+.2f} "
                f"edge={_float_or(score_edge):+.2f} diagnostics="
                f"{comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T"
            )
            for diag in comparison.get("diagnostics", []):
                deep.append(
                    f"  - {diag['label']}: candidate={diag['candidate']:+.2f} "
                    f"baseline={diag['baseline']:+.2f} delta={diag['delta']:+.2f} [{diag['verdict']}]"
                )
        cs = (leader or {}).get("cost_stress", {})
        if cs:
            deep.append(
                "- Cost stress Sharpe: "
                + " | ".join([f"{k}:{v['sharpe']:+.2f}" for k, v in cs.items()])
            )
        subs = (leader or {}).get("subperiods", [])
        if subs:
            deep.append(
                "- Subperiod Sharpe: "
                + " | ".join([f"{s['label']}:{s['sharpe']:+.2f}" for s in subs])
            )

    EVOLUTION_DEEP_DIVE_FILE.write_text("\n".join(deep).strip() + "\n")

    tldr = []
    tldr.append("# Evolution TLDR")
    tldr.append(
        f"- Stop reason: {evo_summary.get('stop_reason')} after {evo_summary.get('generations_executed')} generations."
    )
    if health_lines:
        tldr.append("- Robustness/validity: " + "; ".join(health_lines[:3]) + ".")
    if partial_line:
        tldr.append(f"- Partial run report: {partial_line}.")
    if top_clusters:
        best_train = top_clusters[0]
        tldr.append(
            f"- Best train robust cluster: {best_train.get('cluster_id')} "
            f"(score {_float_or(best_train.get('score')):+.2f}, Sh {_float_or(best_train.get('train_sharpe')):+.2f})."
        )
    if test_rows:
        heldout_report = _build_heldout_report(test_rows)
        leader = heldout_report.get("leader")
        winner = heldout_report.get("winner")
        best_det = heldout_report.get("best_deterministic")
        best_evo = heldout_report.get("best_evolution")
        evo_edge = None if not (best_det and best_evo) else float(best_evo["test_score"] - best_det["test_score"])
        econ_success = bool(
            best_evo
            and best_evo["test_sharpe"] >= ECONOMIC_SHARPE_FLOOR
            and (best_det is None or evo_edge >= ECONOMIC_EDGE_OVER_DETERMINISTIC)
        )
        if winner:
            tldr.append(
                f"- Held-out winner: {winner['cluster_id']} (test score {winner['test_score']:+.2f}, "
                f"test Sharpe {winner['test_sharpe']:+.2f}, edge vs det {heldout_report.get('score_edge_vs_deterministic', 0.0):+.2f})."
            )
        elif leader:
            tldr.append(
                f"- Held-out verdict: no winner yet. Leader is {leader['cluster_id']} "
                f"(score {leader['test_score']:+.2f}, test Sharpe {leader['test_sharpe']:+.2f})."
            )
            tldr.extend([f"- Rule gap: {reason}." for reason in heldout_report.get("reasons", [])])
        else:
            tldr.append("- Held-out verdict: no winner yet.")
        tldr.append(f"- Economic success: {econ_success}.")
        tldr.append(
            f"- Recommended deployment set: top {min(ENSEMBLE_TOP_N, len(test_rows))} held-out variants "
            f"(saved in best_signal_ensemble.py)."
        )
    EVOLUTION_TLDR_FILE.write_text("\n".join(tldr).strip() + "\n")


def evaluate_row_on_test(row):
    short_span, long_span = row.get("short_span", 54), row.get("long_span", 90)
    flipped = bool(row.get("flipped"))
    aux = normalize_aux_params(row.get("params", {}))
    mutation_type = row.get("mutation_type", "plain")
    code_override = row.get("code")
    raw_code = code_override or deterministic_code(row)
    fn = None if code_override else deterministic_signal_factory(short_span, long_span, variant=mutation_type, aux=aux)

    def _run_panel_eval(panel_close, panel_volume, panel_vix, panel_tnx, timeout=25):
        if code_override:
            panel_sig, panel_err = run_signal_code(
                code_override,
                panel_close,
                panel_volume,
                vix_s=panel_vix,
                tnx_s=panel_tnx,
                timeout=timeout,
            )
            if panel_err:
                raise RuntimeError(f"heldout eval code error: {panel_err}")
        else:
            panel_sig = fn(panel_close, panel_volume, panel_vix, panel_tnx)
        if flipped:
            panel_sig = -panel_sig
        panel_metrics = backtest(panel_sig, panel_close)
        return panel_sig, panel_metrics, _oriented_metrics(panel_metrics, flipped=False)

    sig, m, oriented = _run_panel_eval(close_test, volume_test, vix_test, tnx_test, timeout=25)

    def _slice_eval(panel_sig, panel_close, cost_bps=0.0):
        sub_m = backtest(panel_sig, panel_close, cost_bps=cost_bps)
        return sub_m, _oriented_metrics(sub_m, flipped=False)

    beta = oriented["beta"]
    consistency = oriented["consistency"]
    test_sharpe = oriented["sharpe"]
    test_ret = oriented["ann_return"]
    test_dd = oriented["max_dd"]
    test_raw = m.get("sharpe_raw", m.get("sharpe", 0.0))
    bench_spread_sh = m.get("benchmark_spread_sharpe", m.get("excess_sharpe", 0.0))
    wf = []
    n = len(close_test)
    sz = max(n // WF_WINDOWS, 1)
    for w in range(WF_WINDOWS):
        lo = w * sz
        hi = (w + 1) * sz if w < WF_WINDOWS - 1 else n
        try:
            sub_m, sub_oriented = _slice_eval(sig.iloc[lo:hi], close_test.iloc[lo:hi])
        except Exception:
            continue
        wf.append({"window": w, "sharpe": sub_oriented.get("sharpe", sub_m.get("sharpe", 0.0))})
    wf_median, wf_min = wf_summary(wf)
    test_score = _selection_score_safe(test_sharpe, wf_median, consistency, beta, m.get("avg_turnover", 0.0), wf_min=wf_min, max_dd=test_dd)
    test_metric_payload = {
        "train_sharpe": test_sharpe,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "beta": beta,
        "turnover": m.get("avg_turnover", 0.0),
        "consistency": consistency,
        "signal_activity": m.get("signal_activity", 1.0),
        "raw_cs_std": m.get("raw_cs_std", 1.0),
        "raw_long_frac": m.get("raw_long_frac", 1.0),
        "raw_short_frac": m.get("raw_short_frac", 1.0),
    }
    test_robustness = _robustness_score_safe(test_metric_payload)
    cost_stress = {}
    for cost_bps in COST_STRESS_BPS:
        m_cost = backtest(sig, close_test, cost_bps=cost_bps)
        cost_oriented = _oriented_metrics(m_cost, flipped=False)
        cost_stress[f"{int(cost_bps)}bps"] = {
            "sharpe": cost_oriented["sharpe"],
            "ann_return": cost_oriented["ann_return"],
            "max_dd": cost_oriented["max_dd"],
        }

    close_full, volume_full, vix_full, tnx_full = _full_test_panels()
    subperiods = []
    for label, start, end in SUBPERIOD_WINDOWS:
        sub_close = close_full.loc[start:end]
        if len(sub_close) < 60:
            continue
        sub_volume = volume_full.reindex(index=sub_close.index, columns=sub_close.columns)
        sub_vix = vix_full.loc[start:end] if vix_full is not None else None
        sub_tnx = tnx_full.loc[start:end] if tnx_full is not None else None
        try:
            _, sub_m, sub_oriented = _run_panel_eval(sub_close, sub_volume, sub_vix, sub_tnx, timeout=25)
        except Exception:
            continue
        subperiods.append(
            {
                "label": label,
                "start": start,
                "end": end,
                "sharpe": sub_oriented["sharpe"],
                "ann_return": sub_oriented["ann_return"],
                "max_dd": sub_oriented["max_dd"],
                "beta": sub_oriented["beta"],
                "turnover": sub_m["avg_turnover"],
            }
        )

    chronological_holdout = []
    for spec in _build_chronological_window_specs(close_test.index, CHRONOLOGICAL_HOLDOUT_SEGMENTS, min_points=60):
        try:
            sub_m, sub_oriented = _slice_eval(
                sig.iloc[spec["lo"]:spec["hi"]],
                close_test.iloc[spec["lo"]:spec["hi"]],
            )
        except Exception:
            continue
        chronological_holdout.append(
            {
                "label": spec["label"],
                "start": spec["start"],
                "end": spec["end"],
                "sharpe": sub_oriented["sharpe"],
                "ann_return": sub_oriented["ann_return"],
                "max_dd": sub_oriented["max_dd"],
                "beta": sub_oriented["beta"],
                "turnover": sub_m["avg_turnover"],
                "benchmark_spread_sharpe": sub_m.get("benchmark_spread_sharpe", sub_m.get("excess_sharpe", 0.0)),
            }
        )

    rolling_holdout = []
    for spec in _build_rolling_window_specs(close_test.index, ROLLING_HOLDOUT_SEGMENTS, min_points=ROLLING_HOLDOUT_MIN_POINTS):
        try:
            sub_m, sub_oriented = _slice_eval(
                sig.iloc[spec["lo"]:spec["hi"]],
                close_test.iloc[spec["lo"]:spec["hi"]],
            )
        except Exception:
            continue
        rolling_holdout.append(
            {
                "label": spec["label"],
                "start": spec["start"],
                "end": spec["end"],
                "sharpe": sub_oriented["sharpe"],
                "ann_return": sub_oriented["ann_return"],
                "max_dd": sub_oriented["max_dd"],
                "beta": sub_oriented["beta"],
                "turnover": sub_m["avg_turnover"],
                "benchmark_spread_sharpe": sub_m.get("benchmark_spread_sharpe", sub_m.get("excess_sharpe", 0.0)),
            }
        )

    row_source = row.get("source", "deterministic")
    row_iter = _heldout_iter_label(row)
    parent_id = str(row.get("parent_id") or _row_identity_safe(row, default_prefix="parent"))
    hypothesis = row.get("hypothesis")
    if not hypothesis:
        if code_override:
            hypothesis = f"Program evolution {row.get('cluster_id', 'unknown')}"
        else:
            hypothesis = f"Deterministic {mutation_type} EWM {short_span}/{long_span}"

    return {
        "iter": row_iter,
        "parent_id": parent_id,
        "source": row.get("source", "deterministic"),
        "mutation_type": mutation_type,
        "short_span": short_span,
        "long_span": long_span,
        "family": row.get("family", "ewm"),
        "base_family": row.get("base_family", "ewm"),
        "signature": row.get("signature"),
        "cluster_id": row.get("cluster_id", "unknown"),
        "params": aux,
        "hypothesis": hypothesis,
        "train_score": row.get("score", 0.0),
        "test_score": test_score,
        "train_sharpe": row.get("train_sharpe", row.get("sharpe", 0.0)),
        "test_sharpe": test_sharpe,
        "test_consistency": consistency,
        "train_robustness_score": row.get("robustness_score"),
        "test_robustness_score": test_robustness,
        "test_raw": test_raw,
        "test_benchmark_spread_sharpe": bench_spread_sh,
        "test_beta": beta,
        "test_turnover": m["avg_turnover"],
        "test_ret": test_ret,
        "test_dd": test_dd,
        "cost_stress": cost_stress,
        "subperiods": subperiods,
        "chronological_holdout": chronological_holdout,
        "rolling_holdout": rolling_holdout,
        "wf_median": wf_median,
        "wf_min": wf_min,
        "wf_windows": wf,
        "equity": oriented["equity"],
        "code": _oriented_signal_code(raw_code, flipped),
        "raw_code": raw_code,
        "code_is_oriented": True,
        "flipped": flipped,
    }


def evaluate_on_test(top_k=TOP_K):
    if not RUN_HELDOUT_EVAL:
        print("held-out evaluation paused")
        return []
    scope_meta = _runtime_family_scope_lists()
    active_sources = set(scope_meta.get("active_sources", []))
    det = [r for r in _safe_load_search_results_local() if r.get("score") is not None]
    deferred_rows = [r for r in det if _row_matches_deferred_stage_family(r)]
    active_scope_rows = [r for r in det if not _row_matches_deferred_stage_family(r)]
    deterministic = [r for r in active_scope_rows if r.get("source") == "deterministic"]
    evolution = [r for r in active_scope_rows if r.get("source") == "parameter_search_evolution"]
    llm_rows = [r for r in active_scope_rows if r.get("source") == "llm_autoresearch"]
    evolution_champions = [r for r in evolution if r.get("origin") == "deterministic_champion"]
    evolution_rest = [r for r in evolution if r.get("origin") != "deterministic_champion"]
    ignored_other = [r for r in active_scope_rows if r.get("source") not in active_sources]
    picked = []
    seen_sig = set()
    per_cluster = {}
    per_mutation = {}
    picked.extend(_pick_diverse(llm_rows, HELDOUT_MIN_LLM, seen_sig, per_cluster, per_mutation))
    picked.extend(_pick_diverse(evolution_champions, HELDOUT_MIN_EVOLUTION_CHAMPIONS, seen_sig, per_cluster, per_mutation))
    picked.extend(_pick_diverse(evolution_rest, max(0, HELDOUT_MIN_EVOLUTION - len(picked)), seen_sig, per_cluster, per_mutation))
    picked.extend(_pick_diverse(deterministic, HELDOUT_MIN_DETERMINISTIC, seen_sig, per_cluster, per_mutation))
    remaining = max(top_k, HELDOUT_SHORTLIST_K) - len(picked)
    if remaining > 0:
        already = {id(r) for r in picked}
        rest = [r for r in active_scope_rows if id(r) not in already and r.get("source") in active_sources]
        picked.extend(_pick_diverse(rest, remaining, seen_sig, per_cluster, per_mutation))
    det_ranked = picked[:max(top_k, HELDOUT_SHORTLIST_K)]
    print("held-out execution scope:", scope_meta.get("active_scope"))
    print("held-out configured roadmap families:", ", ".join(scope_meta.get("configured", [])))
    print("held-out executed families:", ", ".join(scope_meta.get("executed", [])))
    print("held-out deferred families:", ", ".join(scope_meta.get("deferred", [])))
    if deferred_rows:
        print(f"held-out deferred rows ignored: {len(deferred_rows)}")
    if ignored_other:
        print(f"held-out non-scope rows ignored: {len(ignored_other)}")
    if det_ranked:
        print(f"held-out shortlist: {len(det_ranked)} rows (max {HELDOUT_MAX_PER_CLUSTER} per cluster)")
        source_counts = {}
        for row in det_ranked:
            src = row.get("source", "unknown")
            source_counts[src] = source_counts.get(src, 0) + 1
        print("shortlist sources:", ", ".join([f"{k}={v}" for k, v in sorted(source_counts.items())]))
        evaluated = []
        for r in det_ranked:
            try:
                evaluated.append(evaluate_row_on_test(r))
            except Exception as e:
                print(f"held-out eval skipped {_row_identity_safe(r)}: {e}")
        return sorted(evaluated, key=lambda r: -_float_or(r.get("test_score"), -999.0))
    return []


test_results = evaluate_on_test()
heldout_report = _build_heldout_report(test_results)
ensemble_file = OUT / "best_signal_ensemble.py"


def _write_no_winner_export_guards(heldout_status, reasons=None):
    reasons = list(reasons or [])
    no_winner_guard = (
        "# NO APPROVED WINNER - DEPLOYMENT BLOCKED\n"
        f"# heldout_status={heldout_status}\n"
        "# This file intentionally contains no signal function because this run failed the approved-winner gate.\n"
        "# Rule gaps:\n"
        + "".join(f"# - {reason}\n" for reason in reasons)
        + "\nraise RuntimeError('No approved held-out winner; best_signal.py is non-deployable for this run.')\n"
    )
    BEST_CODE.write_text(no_winner_guard)
    ensemble_file.write_text(no_winner_guard.replace("best_signal.py", "best_signal_ensemble.py"))


if test_results:
    print(f"\n{'iter':>10} {'train_sc':>9} {'test_sc':>8} {'train_Sh':>9} {'test_Sh':>8} {'rob':>6} {'beta':>7} {'to':>6} {'wf_med':>7} {'test_dd':>8}")
    for r in test_results:
        rob = r.get("test_robustness_score", r.get("train_robustness_score"))
        rob_text = "n/a" if rob is None else f"{_float_or(rob):.1f}"
        print(f"{str(r['iter']):>10} {r['train_score']:>+9.2f} {r['test_score']:>+8.2f} {r['train_sharpe']:>+9.2f} {r['test_sharpe']:>+8.2f} {rob_text:>6} {r['test_beta']:>+7.2f} {r['test_turnover']:>6.2f} {r['wf_median']:>+7.2f} {r['test_dd']:>+8.1%}")
    best = heldout_report.get("leader")
    approved_winner = heldout_report.get("winner")
    comparison = heldout_report.get("comparison", {})
    if approved_winner:
        BEST_CODE.write_text(
            f"# objective=market_neutral_net_sharpe  heldout_status={heldout_report.get('status')}  "
            f"iter {approved_winner['iter']}  cluster={approved_winner['cluster_id']}  "
            f"train_score={approved_winner['train_score']:+.2f}  "
            f"test_score={approved_winner['test_score']:+.2f}  train_Sh={approved_winner['train_sharpe']:+.2f}  "
            f"test_Sh={approved_winner['test_sharpe']:+.2f}\n"
            f"# parent={approved_winner['parent_id']}  mutation={approved_winner['mutation_type']}\n"
            f"# winner_rule_edge_vs_det={_float_or(heldout_report.get('score_edge_vs_deterministic'), 0.0):+.2f}  "
            f"diagnostics={comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T\n"
            f"# HYPOTHESIS: {approved_winner['hypothesis']}\n\nimport numpy as np\n\n{approved_winner['code']}\n"
        )
        ensemble_rows = sorted(test_results, key=lambda r: -_float_or(r.get("test_score"), -999.0))[:min(ENSEMBLE_TOP_N, len(test_results))]
        ensemble_file.write_text(_ensemble_code(ensemble_rows))
        print(f"\nheld-out approved winner: iter {approved_winner['iter']}  saved to {BEST_CODE.name}")
        print(
            "approved winner: "
            f"iter {approved_winner['iter']} edge_vs_det={_float_or(heldout_report.get('score_edge_vs_deterministic'), 0.0):+.2f} "
            f"diagnostics={comparison.get('wins', 0)}W/{comparison.get('losses', 0)}L/{comparison.get('ties', 0)}T"
        )
        print(f"ensemble ({len(ensemble_rows)} members) saved to {ensemble_file.name}")
    else:
        _write_no_winner_export_guards(heldout_report.get("status"), heldout_report.get("reasons", []))
        print(f"\nheld-out leader (by composite score): iter {best['iter']}  not exported as deployable code")
        print("approved winner: no winner yet")
        for reason in heldout_report.get("reasons", []):
            print("  rule gap:", reason)
        print(f"deployment guard files written to {BEST_CODE.name} and {ensemble_file.name}")
else:
    _write_no_winner_export_guards(
        "heldout_not_evaluated",
        ["held-out evaluation skipped or no surviving candidates"],
    )
    print("held-out evaluation skipped or no surviving candidates")
    print(f"deployment guard files written to {BEST_CODE.name} and {ensemble_file.name}")

_write_evolution_analysis(test_results)

## Cell 12 â€” Plots

In [ ]:
log     = load_log()
iters   = [e.get("iter", 0) for e in log]
sharpes = [e.get("sharpe") for e in log]

running_best, cur = [], -np.inf
for s in sharpes:
    if s is not None and s > cur:
        cur = s
    running_best.append(cur if cur > -np.inf else np.nan)

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(iters,
           [s if s is not None else np.nan for s in sharpes],
           alpha=0.5, s=40, label="iter Sharpe")
ax.plot(iters, running_best, "-", lw=2, label="running best")
ax.axhline(0, ls="--", color="grey", alpha=0.4)
ax.set_xlabel("Iteration"); ax.set_ylabel("Active Sharpe (train)")
ax.set_title("AutoResearch v2 â€” alpha discovery progress")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(SHARPE_PLOT, dpi=140)
plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 5))
for r in test_results:
    r["equity"].plot(
        ax=ax,
        label=f"iter {r['iter']}  trSh={r['train_sharpe']:+.1f} "
              f"teSh={r['test_sharpe']:+.1f}")
ax.set_ylabel("Equity (active)")
ax.set_title(f"Top-{TOP_K} signals on held-out test (2023â€“2024)")
ax.legend(fontsize=8, loc="best"); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(EQUITY_PLOT, dpi=140)
plt.close(fig)
print("plots saved")


## Cell 13 â€” Experiment summary

In [ ]:

# â”€â”€ Remediation V3: Explicit Artifact States â”€â”€
def _compute_v3_artifact_state(test_results, viable_count, exploration_admissible):
    if exploration_admissible == 0:
        return "exploration_failed_before_heldout"
    if len(test_results) == 0:
        return "exploration_succeeded_but_heldout_skipped"
    # If we reached here with results, the normal winner logic applies
    return None  # fall through to existing logic


log  = load_log()
good = [e for e in log if e.get("sharpe") is not None]
fail = [e for e in log if e.get("error")]
viable = _successful_log_rows(log)
best_train = (max(viable, key=lambda e: e["sharpe"])
              if viable else {"iter": -1, "sharpe": 0, "hypothesis": "(none)"})
best_test  = (max(test_results, key=lambda r: r["test_sharpe"])
              if test_results else None)
runtime_snapshot = sync_runtime_metadata(
    summary_pending=True,
    heldout_result_count=len(test_results),
    successful_backtests=len(viable),
)
actual_model = runtime_snapshot.get("actual_model_id") or "(not loaded)"
hf_meta = runtime_snapshot.get("token_status", {}).get("hf", {})
hf_state = "present" if hf_meta.get("present") else "missing"
hf_source = hf_meta.get("source", "none")
configured_families = runtime_snapshot.get("configured_method_families", ROADMAP_METHOD_FAMILIES)
executed_families = runtime_snapshot.get("executed_method_families", EXECUTED_METHOD_FAMILIES)
deferred_families = runtime_snapshot.get("deferred_method_families", DEFERRED_METHOD_FAMILIES)
active_execution_scope = runtime_snapshot.get("active_execution_scope", ACTIVE_EXECUTION_SCOPE)

md_text = f"""# AutoResearch v2 â€” Momentum Alpha Discovery

**Run:** {RUN_ID} | profile={RUN_PROFILE} | scope={REPORT_SCOPE}
**Benchmark mode:** {BENCHMARK_MODE}
**Configured model:** {CONFIGURED_MODEL_ID}
**Actual model loaded:** {actual_model}
**LLM stage:** enabled={runtime_snapshot.get("llm_stage_enabled")} | loaded={runtime_snapshot.get("llm_stage_loaded")} | executed={runtime_snapshot.get("llm_stage_executed")} | calls={runtime_snapshot.get("llm_calls")}
**HF token:** {hf_state} (source={hf_source})
**Universe:** {len(close_all.columns)} US equities | {close_all.index.min().date()} â†’ {close_all.index.max().date()}
**Train:** through {TRAIN_END} | **Test (held-out):** after

## Method-family scope
- configured roadmap families: {", ".join(configured_families)}
- executed in this stage: {", ".join(executed_families)}
- deferred or not executed in this stage: {", ".join(deferred_families)}
- active execution scope: {active_execution_scope}
- note: LSTM, tabular ML, GBT, and combination studies are deferred in this pure-AutoResearch notebook

## Loop stats
- total batches: {N_BATCHES}
- log entries: {len(log)}
- viable backtests: {len(viable)}
- failures: {len(fail)}
- viable rate: {len(viable)/max(len(log),1):.0%}
- objective mode: `{RUNTIME_STATE.get("objective_mode", OBJECTIVE_MODE)}`
- primary objective: `{RUNTIME_STATE.get("primary_objective_label", PRIMARY_OBJECTIVE_LABEL)}`
- structured candidates recovered: {int(RUNTIME_STATE.get("structured_candidate_count", 0) or 0)}
- repair batches: {int(RUNTIME_STATE.get("repair_batches", 0) or 0)}
- format failures: {int(RUNTIME_STATE.get("format_failures", 0) or 0)}
- duplicate rejections: {int(RUNTIME_STATE.get("duplicate_rejections", 0) or 0)}
- semantic-family rejections: {int(RUNTIME_STATE.get("semantic_family_rejections", 0) or 0)}
- loop status: `{RUNTIME_STATE.get("loop_status", "unknown")}`
- stop reason: `{RUNTIME_STATE.get("loop_stop_reason", "n/a")}`
- stop detail: `{RUNTIME_STATE.get("loop_stop_detail", "n/a")}`
- best train primary Sharpe: **{best_train['sharpe']:+.2f}** (iter {best_train['iter']})

## Output files
- summary: `{SUMMARY_MD.name}`
- runtime metadata: `{RUNTIME_METADATA_FILE.name}`
- run manifest: `{RUN_MANIFEST_FILE.name}`
- research log: `{RESEARCH_LOG.name}`

## Reflection memo
{load_memo()}

## Best on held-out test
"""
if best_test:
    md_text += f"""- iter {best_test['iter']}: train primarySh={best_test['train_sharpe']:+.2f} â†’ test primarySh=**{best_test['test_sharpe']:+.2f}**
- test primary AnnRet: {best_test['test_ret']:+.1%} | test primary DD: {best_test['test_dd']:+.1%} | beta: {best_test['test_beta']:+.2f}
- test benchmark-spread Sh: {best_test.get('test_bench_spread', 0.0):+.2f}
- walk-forward median primarySh: {best_test['wf_median']:+.2f}
- overfit gap (primarySh): {best_test['train_sharpe'] - best_test['test_sharpe']:+.2f}

### Hypothesis
{best_test['hypothesis']}

### Code
```python
{best_test['code']}
```
"""
else:
    md_text += "- no signal survived held-out evaluation.\n"

sync_runtime_metadata(
    summary_pending=False,
    summary_written=True,
    summary_file=str(SUMMARY_MD),
    best_train_iter=best_train.get("iter"),
    best_test_iter=(best_test.get("iter") if best_test else None),
)
SUMMARY_MD.write_text(md_text)
print(md_text)


# === V3 Remediation: Correct top-level artifact states (per plan) ===
try:
    from autoresearch_v3_helpers import classify_v3_run_state

    # exploration_admissible_count = number that passed precheck + were not filtered by staged gate
    exploration_admissible = len([e for e in log if e.get("precheck", {}).get("is_executable") or e.get("_v3_exploration_tier") in ("weak", "promising")])

    heldout_count = len(test_results) if "test_results" in dir() else 0

    # Only treat as explicit skip if the run config deliberately disabled held-out
    explicit_skip = globals().get("RUN_HELDOUT_EVAL", True) is False

    state = classify_v3_run_state(
        exploration_admissible_count=exploration_admissible,
        heldout_result_count=heldout_count,
        winner_rule_passed=False,           # existing winner logic decides this later
        explicit_heldout_skip=explicit_skip,
    )

    update_runtime_state(
        exploration_status=state["exploration_status"],
        heldout_status=state["heldout_status"],
        benchmark_admissible=state["benchmark_admissible"],
        exploration_admissible_count=exploration_admissible,
        heldout_eligible_count=heldout_count,
        artifact_integrity_error=False,
    )
except Exception as _e:
    try:
        update_runtime_state(artifact_integrity_error=True)
    except:
        pass

